In [12]:
def calculate_center_of_mass(c0_image_path, c3_image_path, output_base):
    """
    Use Li thresholding on the _C_0 image to apply the boundaries of the white thresholded object
    to the thresholded _C_3 image and calculate the center of mass of the enclosed _C_3 object.
    A blue crosshair is drawn on the overlay image to signify the center of mass.

    Parameters
    ----------
    c0_image_path : str
        Path to the _C_0 image.
    c3_image_path : str
        Path to the _C_3 image.
    output_base : str
        Base path for saving the overlay image with the blue crosshair.

    Returns
    -------
    tuple or None
        (x, y) coordinates of the center of mass of the masked thresholded _C_3 object, or None if no object is found.
    """
    import cv2
    import numpy as np
    from skimage.measure import regionprops, label as skimage_label
    from skimage.filters import threshold_li

    # Load the _C_0 image and apply Li threshold
    c0_image = cv2.imread(c0_image_path, cv2.IMREAD_GRAYSCALE)
    if c0_image is None:
        raise ValueError(f"Error: Unable to load _C_0 image at {c0_image_path}")
    
    try:
        li_thresh_c0 = threshold_li(c0_image)
        c0_binary = c0_image > li_thresh_c0
        c0_binary = (c0_binary * 255).astype(np.uint8)
    except Exception as e:
        print(f"Error: Li thresholding failed for _C_0 image: {e}")
        return None

    # Load the _C_3 image and apply Li threshold
    c3_image = cv2.imread(c3_image_path, cv2.IMREAD_GRAYSCALE)
    if c3_image is None:
        raise ValueError(f"Error: Unable to load _C_3 image at {c3_image_path}")
    
    try:
        li_thresh_c3 = threshold_li(c3_image)
        c3_binary = c3_image > li_thresh_c3
        c3_binary = (c3_binary * 255).astype(np.uint8)
    except Exception as e:
        print(f"Error: Li thresholding failed for _C_3 image: {e}")
        return None

    # Apply the mask from the _C_0 image to the thresholded _C_3 image
    masked_c3_image = cv2.bitwise_and(c3_binary, c3_binary, mask=c0_binary)

    # Label connected components in the masked thresholded _C_3 image
    labeled_image = skimage_label(masked_c3_image)

    # Check if any objects are found
    props = regionprops(labeled_image)
    if not props:
        print(f"Warning: No objects found in masked thresholded _C_3 image {c3_image_path}. Skipping this image.")
        return None

    # Calculate the center of mass of the largest connected component
    largest_region = max(props, key=lambda x: x.area)
    center_of_mass = largest_region.centroid  # (y, x) format
    center_x, center_y = int(center_of_mass[1]), int(center_of_mass[0])  # Convert to (x, y)

    # Create an overlay image with the blue crosshair
    overlay = cv2.cvtColor(c3_image, cv2.COLOR_GRAY2BGR)  # Convert to BGR for color overlay
    crosshair_color = (255, 0, 0)  # Blue color (BGR format)
    crosshair_thickness = 2
    crosshair_length = 10

    # Draw the horizontal and vertical lines of the crosshair
    cv2.line(overlay, (center_x - crosshair_length, center_y), (center_x + crosshair_length, center_y), crosshair_color, crosshair_thickness)
    cv2.line(overlay, (center_x, center_y - crosshair_length), (center_x, center_y + crosshair_length), crosshair_color, crosshair_thickness)

    # Save the overlay image with the blue crosshair
    overlay_output_path = f"{output_base}_C3_center_of_mass_overlay.png"
    cv2.imwrite(overlay_output_path, overlay)

    return center_x, center_y


In [25]:
def generate_skeletonize_representatives(image_folder):
    """
    Process ALL images in `image_folder` and generate a 1x4 side-by-side panel per image:
      [Original | skeletonize | thin | medial_axis]

    Notes
    -----
    - Uses Li thresholding ONLY to binarize each image prior to skeletonization.
    - Saves outputs to: <image_folder>/Skeletonize_Representative_Images_LI
    - Reads common image formats: .tif/.tiff/.png/.jpg/.jpeg/.bmp
    """
    import os
    import matplotlib.pyplot as plt
    from skimage import io, img_as_float
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize, thin, medial_axis

    valid_ext = (".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp")

    if not os.path.isdir(image_folder):
        raise FileNotFoundError(f"image_folder not found: {image_folder}")

    output_folder = os.path.join(image_folder, "Skeletonize_Representative_Images")
    os.makedirs(output_folder, exist_ok=True)

    all_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(valid_ext)])
    if len(all_files) == 0:
        raise ValueError(f"No images found in {image_folder} matching extensions {valid_ext}")

    saved_files = []

    for fname in all_files:
        in_path = os.path.join(image_folder, fname)

        img = io.imread(in_path, as_gray=True)
        img = img_as_float(img)

        # Li threshold ONLY
        t = threshold_li(img)
        mask = img > t

        skel = skeletonize(mask)
        thin_skel = thin(mask)
        medial_skel, _ = medial_axis(mask, return_distance=True)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        ax0, ax1, ax2, ax3 = axes

        ax0.imshow(img, cmap="gray")
        ax0.set_title("Original", fontweight="bold")
        ax0.axis("off")

        ax1.imshow(skel, cmap="gray")
        ax1.set_title("skeletonize()", fontweight="bold")
        ax1.axis("off")

        ax2.imshow(thin_skel, cmap="gray")
        ax2.set_title("thin()", fontweight="bold")
        ax2.axis("off")

        ax3.imshow(medial_skel, cmap="gray")
        ax3.set_title("medial_axis()", fontweight="bold")
        ax3.axis("off")

        fig.suptitle(f"{fname} | Li t={t:.4f}", y=1.03, fontsize=11)
        plt.tight_layout()

        base, _ = os.path.splitext(fname)
        out_path = os.path.join(output_folder, f"{base}_skeleton_compare.png")
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig)

        saved_files.append(out_path)

    return {"output_folder": output_folder, "saved_files": saved_files}

generate_skeletonize_representatives('/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI')

{'output_folder': '/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI/Skeletonize_Representative_Images',
 'saved_files': ['/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI/Skeletonize_Representative_Images/Tri_5ugLPS_01_1_C_0_skeleton_compare.png',
  '/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI/Skeletonize_Representative_Images/Tri_5ugLPS_01_1_C_3_skeleton_compare.png',
  '/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI/Skeletonize_Representative_Images/Tri_5ugLPS_01_2_C_0_skeleton_compare.png',
  '/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI/Skeletonize_Representative_Images/Tri_5ugLPS_01_2_C_3_skeleton_compare.png',
  '/Users/allisonpickle/Desktop/Microglia_DAPI_for Sholl/All/Isolated_Microglia_DAPI/Skeletonize_Representative_Images/Tri_5ugLPS_02_1_C_0_skeleton_compare.png',
  '/Users/allisonpickle/Desktop/Microg

In [ ]:
def Sholl_Compare_Skeleton_Methods(
    input_folder,
    output_folder_name="Sholl Analysis - Skeleton Compare",
    start_radius_microns=0,
    step_radius_microns=2.5,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
):
    """
    Run Sholl analysis using Li thresholding + multiple scikit-image skeletonization methods
    for comparison on each matching (_C_0.tiff, _C_3.tiff) pair in `input_folder`.

    Skeleton methods compared
    -------------------------
    - skeletonize
    - thin
    - medial_axis

    Expected inputs
    ---------------
    For each cell, `input_folder` contains:
      <prefix>_C_0.tiff  (DAPI/soma mask; used to compute soma center + optional overlay background)
      <prefix>_C_3.tiff  (process channel; skeletonization source for Sholl intersections)

    Outputs
    -------
    Creates a folder `<input_folder>/<output_folder_name>` containing:
      - Parameters text file
      - Per-image overlay PNGs for each skeleton method
      - Per-image Sholl curve plots comparing methods
      - One Excel file with:
          * ImageLevel sheet (per image, per method, per radius intersections)
          * Summary sheet (AUC, max intersections, peak radius, per image/per method)
          * MethodMeans sheet (mean curve across images per method)

    Notes
    -----
    - Uses your existing `calculate_center_of_mass(c0_image_path, c3_image_path, output_base)`
      to obtain soma coordinates (x, y).
    - Applies Li thresholding to the _C_3 image to create a binary mask for skeletonization.
    - Intersection counting matches your current approach: counts unique connected components
      of skeleton that overlap a 2-pixel-thick circular band (radius±1 px).
    """
    import os
    from datetime import datetime
    import numpy as np
    import pandas as pd
    import cv2
    import matplotlib.pyplot as plt
    from scipy.ndimage import label as ndi_label
    from scipy.interpolate import interp1d
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize, thin, medial_axis

    # -----------------------------
    # Helpers
    # -----------------------------
    def _list_pairs(folder):
        c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(folder) if fn.endswith("_C_3.tiff")}
        c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(folder) if fn.endswith("_C_0.tiff")}
        keys = sorted(list(c3.keys() & c0.keys()))
        return keys, c0, c3

    def _binary_from_li(gray_u8):
        t = threshold_li(gray_u8)
        return (gray_u8 > t), float(t)

    def _compute_intersections(skel_bool, center_x, center_y, radii_px):
        # label skeleton ONCE
        labeled, nfeat = ndi_label(skel_bool.astype(np.uint8))

        rr, cc = np.ogrid[:skel_bool.shape[0], :skel_bool.shape[1]]
        dist = np.sqrt((rr - center_y) ** 2 + (cc - center_x) ** 2)

        distances_um = []
        counts = []

        for r in radii_px:
            band = (dist >= (r - 1)) & (dist <= (r + 1))

            # count unique components that touch the band
            if nfeat == 0:
                count = 0
            else:
                labels_touching = np.unique(labeled[band])
                labels_touching = labels_touching[labels_touching != 0]
                count = int(labels_touching.size)

            distances_um.append(r * scale_microns_per_pixel)
            counts.append(count)

        return distances_um, counts

    def _save_overlay(gray_u8, skel_bool, center_xy, radii_px, out_png):
        overlay = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2BGR)
        overlay[skel_bool] = (0, 0, 255)  # red skeleton

        cx, cy = center_xy
        for r in radii_px:
            cv2.circle(overlay, (cx, cy), int(r), (0, 255, 0), 1)

            # mark intersection pixels (pink) for this band
            rr, cc = np.ogrid[:skel_bool.shape[0], :skel_bool.shape[1]]
            dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)
            band = (dist >= (r - 1)) & (dist <= (r + 1))
            coords = np.argwhere(skel_bool & band)
            for y, x in coords:
                cv2.circle(overlay, (int(x), int(y)), 2, (255, 105, 180), -1)

        cv2.imwrite(out_png, overlay)

    # -----------------------------
    # Main
    # -----------------------------
    if not os.path.isdir(input_folder):
        raise FileNotFoundError(f"input_folder not found: {input_folder}")

    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    # Save parameters
    params_file = os.path.join(output_folder, "Sholl_Compare_Skeleton_Methods_Parameters.txt")
    with open(params_file, "w") as f:
        f.write("=" * 70 + "\n")
        f.write("SHOLL ANALYSIS - SKELETON METHOD COMPARISON\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"Analysis Date/Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input Folder: {input_folder}\n")
        f.write(f"Output Folder: {output_folder}\n\n")
        f.write("RADIUS PARAMETERS\n")
        f.write("-" * 70 + "\n")
        f.write(f"Start radius (µm): {start_radius_microns}\n")
        f.write(f"Step radius (µm):  {step_radius_microns}\n")
        f.write(f"Max radius (µm):   {max_radius_microns}\n\n")
        f.write("IMAGE/SCALE\n")
        f.write("-" * 70 + "\n")
        f.write(f"Scale factor: {scale_microns_per_pixel} µm/pixel\n")
        f.write("Thresholding: Li (on _C_3)\n")
        f.write("Skeleton methods: skeletonize, thin, medial_axis\n")

    keys, c0_map, c3_map = _list_pairs(input_folder)
    if not keys:
        raise ValueError("No matching _C_0.tiff / _C_3.tiff pairs found in input_folder.")

    # Build radii in pixels
    start_px = int(round(start_radius_microns / scale_microns_per_pixel))
    step_px = int(round(step_radius_microns / scale_microns_per_pixel))
    max_px = int(round(max_radius_microns / scale_microns_per_pixel))
    radii_px = list(range(start_px, max_px + 1, step_px))

    methods = {
        "skeletonize": lambda m: skeletonize(m),
        "thin": lambda m: thin(m),
        "medial_axis": lambda m: medial_axis(m, return_distance=False),
    }

    # Storage
    imagelevel_rows = []
    summary_rows = []
    curves_by_method = {m: [] for m in methods.keys()}  # list of (dist_um, counts) per image

    for key in keys:
        c0_path = os.path.join(input_folder, c0_map[key])
        c3_path = os.path.join(input_folder, c3_map[key])

        output_base = os.path.join(output_folder, os.path.splitext(c0_map[key])[0])

        soma_xy = calculate_center_of_mass(c0_path, c3_path, output_base)
        if soma_xy is None:
            continue
        cx, cy = soma_xy

        # Load _C_3 as skeleton input
        c3_img = cv2.imread(c3_path, cv2.IMREAD_GRAYSCALE)
        if c3_img is None:
            continue

        # Li threshold on _C_3
        mask, li_t = _binary_from_li(c3_img)

        # Per-image plot comparing methods
        fig = plt.figure(figsize=(9, 6))

        for method_name, skel_fn in methods.items():
            skel = skel_fn(mask).astype(bool)

            dist_um, counts = _compute_intersections(skel, cx, cy, radii_px)
            curves_by_method[method_name].append((dist_um, counts))

            # Save overlay per method
            overlay_png = f"{output_base}_{method_name}_sholl_overlay.png"
            _save_overlay(c3_img, skel, (cx, cy), radii_px, overlay_png)

            # Save to image-level rows
            for d, c in zip(dist_um, counts):
                imagelevel_rows.append(
                    {
                        "Key": key,
                        "Image_C0": c0_map[key],
                        "Image_C3": c3_map[key],
                        "Method": method_name,
                        "Li_Threshold_C3": li_t,
                        "Distance_um": d,
                        "Intersections": c,
                    }
                )

            # Summary stats per image/method
            auc = float(np.trapz(counts, dist_um))
            max_int = int(np.max(counts)) if len(counts) else 0
            peak_idx = int(np.argmax(counts)) if len(counts) else 0
            peak_radius = float(dist_um[peak_idx]) if len(dist_um) else np.nan

            summary_rows.append(
                {
                    "Key": key,
                    "Image_C0": c0_map[key],
                    "Image_C3": c3_map[key],
                    "Method": method_name,
                    "Li_Threshold_C3": li_t,
                    "AUC": auc,
                    "Max_Intersections": max_int,
                    "Peak_Radius_um": peak_radius,
                }
            )

            plt.plot(dist_um, counts, marker="o", linewidth=2, label=method_name)

        plt.title(f"Sholl Curve Comparison | {key}")
        plt.xlabel("Distance from soma (µm)")
        plt.ylabel("Intersections")
        plt.grid(True, alpha=0.3)
        plt.legend()
        curve_png = f"{output_base}_sholl_curve_compare.png"
        plt.tight_layout()
        plt.savefig(curve_png, dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Build outputs
    imagelevel_df = pd.DataFrame(imagelevel_rows)
    summary_df = pd.DataFrame(summary_rows)

    if imagelevel_df.empty:
        raise ValueError("No Sholl results generated (no valid images or soma centers).")

    # Mean curves per method (resample to common distances)
    resampled_dist = np.array([r * scale_microns_per_pixel for r in radii_px], dtype=float)
    methodmean_rows = []
    for method_name, curves in curves_by_method.items():
        if not curves:
            continue
        # curves already on same grid; still guard
        mat = np.array([c for (_, c) in curves], dtype=float)
        mean_curve = np.mean(mat, axis=0)
        sem_curve = np.std(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0]) if mat.shape[0] > 1 else np.zeros_like(mean_curve)

        for d, mval, sval in zip(resampled_dist, mean_curve, sem_curve):
            methodmean_rows.append(
                {"Method": method_name, "Distance_um": float(d), "Mean_Intersections": float(mval), "SEM": float(sval)}
            )

    methodmeans_df = pd.DataFrame(methodmean_rows)

    # Save combined mean curve plot
    plt.figure(figsize=(9, 6))
    for method_name in methodmeans_df["Method"].unique():
        sub = methodmeans_df[methodmeans_df["Method"] == method_name]
        plt.plot(sub["Distance_um"], sub["Mean_Intersections"], marker="o", linewidth=2, label=method_name)
    plt.title("Mean Sholl Curves by Skeleton Method")
    plt.xlabel("Distance from soma (µm)")
    plt.ylabel("Mean intersections")
    plt.grid(True, alpha=0.3)
    plt.legend()
    mean_png = os.path.join(output_folder, "Mean_Sholl_Curves_by_Method.png")
    plt.tight_layout()
    plt.savefig(mean_png, dpi=300, bbox_inches="tight")
    plt.close()

    # Save Excel
    excel_path = os.path.join(output_folder, "Sholl_Compare_Skeleton_Methods.xlsx")
    with pd.ExcelWriter(excel_path) as writer:
        imagelevel_df.to_excel(writer, sheet_name="ImageLevel", index=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        methodmeans_df.to_excel(writer, sheet_name="MethodMeans", index=False)

    print(f"\nComplete. Outputs saved to: {output_folder}")
    print(f"Excel: {excel_path}")
    print(f"Mean curve plot: {mean_png}")

    return {
        "output_folder": output_folder,
        "excel_path": excel_path,
        "imagelevel_df": imagelevel_df,
        "summary_df": summary_df,
        "methodmeans_df": methodmeans_df,
    }

In [1]:
def delete_orphan_c3_images(folder_path):
    """
    Scan `folder_path` for paired images:
      <prefix>_C_0.tiff  and  <prefix>_C_3.tiff

    If a _C_3.tiff exists without its corresponding _C_0.tiff, delete the _C_3.tiff.

    Returns
    -------
    dict with:
      - deleted: list of deleted file paths
      - kept: list of _C_3.tiff paths that had a matching _C_0.tiff
      - missing_c0: list of expected _C_0.tiff paths that were missing
    """
    import os

    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f"Folder not found: {folder_path}")

    files = os.listdir(folder_path)

    c0 = {fn.split("_C_", 1)[0]: fn for fn in files if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in files if fn.endswith("_C_3.tiff")}

    deleted = []
    kept = []
    missing_c0 = []

    for key, c3_name in sorted(c3.items()):
        if key not in c0:
            c3_path = os.path.join(folder_path, c3_name)
            expected_c0_path = os.path.join(folder_path, f"{key}_C_0.tiff")

            # delete orphan C3
            os.remove(c3_path)
            deleted.append(c3_path)
            missing_c0.append(expected_c0_path)
        else:
            kept.append(os.path.join(folder_path, c3_name))

    return {"deleted": deleted, "kept": kept, "missing_c0": missing_c0}


delete_orphan_c3_images('/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI')

{'deleted': ['/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/100ng_LPS_04_5_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/100ng_LPS_07_1_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/100ng_LPS_07_3_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_01_2_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_07_1_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_11_3_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_11_5_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_12_3_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_12_4_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/500ng_LPS_12_5_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI/5ng_LPS_04_1_C_3.tiff',
  '/Users/allisonpickle/Desktop/All/Isolated_

In [5]:
def calculate_center_of_mass(c0_image_path, c3_image_path, output_base):
    """
    Use Li thresholding on the _C_0 image to apply the boundaries of the white thresholded object
    to the thresholded _C_3 image and calculate the center of mass of the enclosed _C_3 object.
    A blue crosshair is drawn on the overlay image to signify the center of mass.

    Parameters
    ----------
    c0_image_path : str
        Path to the _C_0 image.
    c3_image_path : str
        Path to the _C_3 image.
    output_base : str
        Base path for saving the overlay image with the blue crosshair.

    Returns
    -------
    tuple or None
        (x, y) coordinates of the center of mass of the masked thresholded _C_3 object, or None if no object is found.
    """
    import cv2
    import numpy as np
    from skimage.measure import regionprops, label as skimage_label
    from skimage.filters import threshold_li

    # Load the _C_0 image and apply Li threshold
    c0_image = cv2.imread(c0_image_path, cv2.IMREAD_GRAYSCALE)
    if c0_image is None:
        raise ValueError(f"Error: Unable to load _C_0 image at {c0_image_path}")
    
    try:
        li_thresh_c0 = threshold_li(c0_image)
        c0_binary = c0_image > li_thresh_c0
        c0_binary = (c0_binary * 255).astype(np.uint8)
    except Exception as e:
        print(f"Error: Li thresholding failed for _C_0 image: {e}")
        return None

    # Load the _C_3 image and apply Li threshold
    c3_image = cv2.imread(c3_image_path, cv2.IMREAD_GRAYSCALE)
    if c3_image is None:
        raise ValueError(f"Error: Unable to load _C_3 image at {c3_image_path}")
    
    try:
        li_thresh_c3 = threshold_li(c3_image)
        c3_binary = c3_image > li_thresh_c3
        c3_binary = (c3_binary * 255).astype(np.uint8)
    except Exception as e:
        print(f"Error: Li thresholding failed for _C_3 image: {e}")
        return None

    # Apply the mask from the _C_0 image to the thresholded _C_3 image
    masked_c3_image = cv2.bitwise_and(c3_binary, c3_binary, mask=c0_binary)

    # Label connected components in the masked thresholded _C_3 image
    labeled_image = skimage_label(masked_c3_image)

    # Check if any objects are found
    props = regionprops(labeled_image)
    if not props:
        print(f"Warning: No objects found in masked thresholded _C_3 image {c3_image_path}. Skipping this image.")
        return None

    # Calculate the center of mass of the largest connected component
    largest_region = max(props, key=lambda x: x.area)
    center_of_mass = largest_region.centroid  # (y, x) format
    center_x, center_y = int(center_of_mass[1]), int(center_of_mass[0])  # Convert to (x, y)

    # Create an overlay image with the blue crosshair
    overlay = cv2.cvtColor(c3_image, cv2.COLOR_GRAY2BGR)  # Convert to BGR for color overlay
    crosshair_color = (255, 0, 0)  # Blue color (BGR format)
    crosshair_thickness = 2
    crosshair_length = 10

    # Draw the horizontal and vertical lines of the crosshair
    cv2.line(overlay, (center_x - crosshair_length, center_y), (center_x + crosshair_length, center_y), crosshair_color, crosshair_thickness)
    cv2.line(overlay, (center_x, center_y - crosshair_length), (center_x, center_y + crosshair_length), crosshair_color, crosshair_thickness)

    # Save the overlay image with the blue crosshair
    overlay_output_path = f"{output_base}_C3_center_of_mass_overlay.png"
    cv2.imwrite(overlay_output_path, overlay)

    return center_x, center_y
def Process_Sholl_Skeleton_Comparison(
    input_folder,
    start_radius_microns,
    step_radius_microns,
    output_folder_name="Sholl Analysis - Skeleton Comparison",
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
):
    """
    Sholl analysis comparing scikit-image skeletonization methods.

    Excel output includes:
      - ImageLevel, Summary, MethodMeans
      - one sheet per method (intersections wide) + one sheet per method (summary)
      - NEW: AUC sheets:
          * AUC_All (long)
          * AUC_Wide (one column per method)
          * AUC_<method> (optional per-method AUC-only sheets)
    """
    import os
    from datetime import datetime

    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import label as ndi_label
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize, thin, medial_axis

    if not os.path.isdir(input_folder):
        raise FileNotFoundError(f"input_folder not found: {input_folder}")

    if start_radius_microns <= 0:
        raise ValueError("start_radius_microns must be > 0")
    if step_radius_microns <= 0:
        raise ValueError("step_radius_microns must be > 0")
    if max_radius_microns <= start_radius_microns:
        raise ValueError("max_radius_microns must be > start_radius_microns")

    def _safe_float_tag(x, decimals=2):
        s = f"{float(x):.{decimals}f}"
        return s.replace(".", "p")

    def _excel_sheet_name(name, max_len=31):
        bad = [":", "\\", "/", "?", "*", "[", "]"]
        for ch in bad:
            name = name.replace(ch, "_")
        return name[:max_len]

    output_folder_name = f"{output_folder_name}_startR_{_safe_float_tag(start_radius_microns)}um"
    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(list(c0.keys() & c3.keys()))
    if not keys:
        raise ValueError("No matching _C_0.tiff / _C_3.tiff pairs found in input_folder.")

    start_px = int(round(start_radius_microns / scale_microns_per_pixel))
    step_px = int(round(step_radius_microns / scale_microns_per_pixel))
    max_px = int(round(max_radius_microns / scale_microns_per_pixel))
    radii_px = list(range(start_px, max_px + 1, step_px))
    radii_um = [r * scale_microns_per_pixel for r in radii_px]

    methods = {
        "skeletonize": lambda m: skeletonize(m).astype(bool),
        "thin": lambda m: thin(m).astype(bool),
        "medial_axis": lambda m: medial_axis(m, return_distance=False).astype(bool),
    }

    def _li_mask(gray_u8):
        t = float(threshold_li(gray_u8))
        return (gray_u8 > t), t

    def _count_intersections(skel_bool, cx, cy):
        labeled, nfeat = ndi_label(skel_bool.astype(np.uint8))

        rr, cc = np.ogrid[:skel_bool.shape[0], :skel_bool.shape[1]]
        dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)

        counts = []
        for r in radii_px:
            band = (dist >= (r - 1)) & (dist <= (r + 1))
            if nfeat == 0:
                counts.append(0)
            else:
                touching = np.unique(labeled[band])
                touching = touching[touching != 0]
                counts.append(int(touching.size))
        return counts

    def _save_overlay(gray_u8, skel_bool, cx, cy, out_png):
        overlay = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2BGR)
        overlay[skel_bool] = (0, 0, 255)

        rr, cc = np.ogrid[:skel_bool.shape[0], :skel_bool.shape[1]]
        dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)

        for r in radii_px:
            cv2.circle(overlay, (int(cx), int(cy)), int(r), (0, 255, 0), 1)
            band = (dist >= (r - 1)) & (dist <= (r + 1))
            coords = np.argwhere(skel_bool & band)
            for y, x in coords:
                cv2.circle(overlay, (int(x), int(y)), 2, (255, 105, 180), -1)

        cv2.imwrite(out_png, overlay)

    imagelevel_rows = []
    summary_rows = []
    curves_by_method = {m: [] for m in methods.keys()}

    params_file = os.path.join(output_folder, "Parameters.txt")
    with open(params_file, "w") as f:
        f.write("=" * 70 + "\n")
        f.write("SHOLL ANALYSIS - SKELETON METHOD COMPARISON\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input folder: {input_folder}\n")
        f.write(f"Output folder: {output_folder}\n\n")
        f.write(f"start_radius_microns: {start_radius_microns}\n")
        f.write(f"step_radius_microns:  {step_radius_microns}\n")
        f.write(f"max_radius_microns:   {max_radius_microns}\n")
        f.write(f"scale_microns_per_pixel: {scale_microns_per_pixel}\n")
        f.write("Center: from _C_3 (calculate_center_of_mass)\n")
        f.write("Sholl skeleton/intersections: on _C_0 only (Li threshold)\n")
        f.write(f"methods: {', '.join(methods.keys())}\n")

    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])
        output_base = os.path.join(output_folder, os.path.splitext(c0[key])[0])

        soma_xy = calculate_center_of_mass(c0_path, c3_path, output_base)
        if soma_xy is None:
            continue
        cx, cy = soma_xy

        c0_img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if c0_img is None:
            print(f"Warning: unable to load {c0_path}, skipping.")
            continue

        mask_c0, li_t_c0 = _li_mask(c0_img)

        plt.figure(figsize=(9, 6))

        for method_name, fn in methods.items():
            skel = fn(mask_c0)
            counts = _count_intersections(skel, cx, cy)
            curves_by_method[method_name].append(counts)

            overlay_png = f"{output_base}_C0_{method_name}_overlay.png"
            _save_overlay(c0_img, skel, cx, cy, overlay_png)

            for d_um, c_int in zip(radii_um, counts):
                imagelevel_rows.append(
                    {
                        "Key": key,
                        "Image_C0": c0[key],
                        "Image_C3": c3[key],
                        "Method": method_name,
                        "Li_Threshold_C0": li_t_c0,
                        "Distance_um": float(d_um),
                        "Intersections": int(c_int),
                    }
                )

            auc = float(np.trapz(counts, radii_um))
            max_int = int(np.max(counts)) if counts else 0
            peak_idx = int(np.argmax(counts)) if counts else 0
            peak_radius_um = float(radii_um[peak_idx]) if counts else np.nan

            summary_rows.append(
                {
                    "Key": key,
                    "Image_C0": c0[key],
                    "Image_C3": c3[key],
                    "Method": method_name,
                    "Li_Threshold_C0": li_t_c0,
                    "AUC": auc,
                    "Max_Intersections": max_int,
                    "Peak_Radius_um": peak_radius_um,
                }
            )

            plt.plot(radii_um, counts, marker="o", linewidth=2, label=method_name)

        plt.title(f"Sholl Curve Comparison (on C0) | {key}")
        plt.xlabel("Distance from soma (µm)")
        plt.ylabel("Intersections")
        plt.grid(True, alpha=0.3)
        plt.legend()
        curve_png = f"{output_base}_C0_curve_compare.png"
        plt.tight_layout()
        plt.savefig(curve_png, dpi=300, bbox_inches="tight")
        plt.close()

    imagelevel_df = pd.DataFrame(imagelevel_rows)
    summary_df = pd.DataFrame(summary_rows)
    if imagelevel_df.empty:
        raise ValueError("No results generated. Check that images exist and center-of-mass is found.")

    methodmean_rows = []
    for method_name, all_counts in curves_by_method.items():
        if not all_counts:
            continue
        mat = np.asarray(all_counts, dtype=float)
        mean_curve = np.mean(mat, axis=0)
        sem_curve = np.std(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0]) if mat.shape[0] > 1 else np.zeros_like(mean_curve)

        for d_um, mval, sval in zip(radii_um, mean_curve, sem_curve):
            methodmean_rows.append(
                {
                    "Method": method_name,
                    "Distance_um": float(d_um),
                    "Mean_Intersections": float(mval),
                    "SEM": float(sval),
                    "N_Images": int(mat.shape[0]),
                }
            )

    methodmeans_df = pd.DataFrame(methodmean_rows)

    plt.figure(figsize=(9, 6))
    for method_name in methodmeans_df["Method"].unique():
        sub = methodmeans_df[methodmeans_df["Method"] == method_name]
        plt.plot(sub["Distance_um"], sub["Mean_Intersections"], marker="o", linewidth=2, label=method_name)
    plt.title("Mean Sholl Curves by Skeleton Method (on C0)")
    plt.xlabel("Distance from soma (µm)")
    plt.ylabel("Mean intersections")
    plt.grid(True, alpha=0.3)
    plt.legend()
    mean_png = os.path.join(output_folder, "Mean_Sholl_Curves_by_Method_C0.png")
    plt.tight_layout()
    plt.savefig(mean_png, dpi=300, bbox_inches="tight")
    plt.close()

    # per-method sheets (wide intersections + summary)
    method_sheets = {}
    for method_name in methods.keys():
        sub = imagelevel_df[imagelevel_df["Method"] == method_name].copy()
        wide = (
            sub.pivot_table(
                index=["Key", "Image_C0", "Image_C3"],
                columns="Distance_um",
                values="Intersections",
                aggfunc="first",
            )
            .reset_index()
        )
        wide.columns = [col if isinstance(col, str) else f"{float(col):.3f}um" for col in wide.columns]
        method_sheets[method_name] = wide

    method_summary_sheets = {}
    for method_name in methods.keys():
        method_summary_sheets[method_name] = summary_df[summary_df["Method"] == method_name].copy()

    # --- NEW: AUC-only sheets ---
    auc_all_df = summary_df[
        ["Key", "Image_C0", "Image_C3", "Method", "AUC"]
    ].copy()

    auc_wide_df = (
        auc_all_df.pivot_table(
            index=["Key", "Image_C0", "Image_C3"],
            columns="Method",
            values="AUC",
            aggfunc="first",
        )
        .reset_index()
    )

    auc_by_method_sheets = {}
    for method_name in methods.keys():
        auc_by_method_sheets[method_name] = auc_all_df[auc_all_df["Method"] == method_name].copy()

    # --- Excel ---
    excel_path = os.path.join(output_folder, "Sholl_Skeleton_Comparison.xlsx")
    with pd.ExcelWriter(excel_path) as writer:
        imagelevel_df.to_excel(writer, sheet_name="ImageLevel", index=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        methodmeans_df.to_excel(writer, sheet_name="MethodMeans", index=False)

        for method_name, df_wide in method_sheets.items():
            sheet = _excel_sheet_name(f"{method_name}_intersections")
            df_wide.to_excel(writer, sheet_name=sheet, index=False)

        for method_name, df_sum in method_summary_sheets.items():
            sheet = _excel_sheet_name(f"{method_name}_summary")
            df_sum.to_excel(writer, sheet_name=sheet, index=False)

        # NEW AUC sheets
        auc_all_df.to_excel(writer, sheet_name="AUC_All", index=False)
        auc_wide_df.to_excel(writer, sheet_name="AUC_Wide", index=False)

        for method_name, df_auc in auc_by_method_sheets.items():
            sheet = _excel_sheet_name(f"AUC_{method_name}")
            df_auc.to_excel(writer, sheet_name=sheet, index=False)

    print(f"Done. Results in: {output_folder}")
    return {
        "output_folder": output_folder,
        "excel_path": excel_path,
        "imagelevel_df": imagelevel_df,
        "summary_df": summary_df,
        "methodmeans_df": methodmeans_df,
        "auc_all_df": auc_all_df,
        "auc_wide_df": auc_wide_df,
    }
Process_Sholl_Skeleton_Comparison(
    input_folder='/Users/allisonpickle/Desktop/LPS/3 BioReps Sholl/All_Bio_Reps/Isolated_Microglia_DAPI',
    output_folder_name="0_Sholl Analysis_Skeleton Comparison",
    start_radius_microns=9.89,
    step_radius_microns=5,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
)

FileNotFoundError: input_folder not found: /Users/allisonpickle/Desktop/LPS/3 BioReps Sholl/All_Bio_Reps/Isolated_Microglia_DAPI

Sholl analysis for different step size

In [7]:
def Process_Sholl_StandardSkeleton(
    input_folder,
    start_radius_microns,
    step_radius_microns,
    output_folder_name="Sholl Analysis - Standard Skeleton",
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    overlay_ring_interval_microns=10,
):
    """
    Sholl analysis using ONE standard skeletonization (skimage.morphology.skeletonize) on C0.

    step_radius_microns:
      - a positive number -> rings every N microns
      - "continuous" or None -> continuous sampling (every 1-pixel radius)

    Splits the mean Sholl curve by Treatment inferred from the SECOND
    underscore-delimited token in the filename stem.

    Example expected filename stem:
      Tri_Ctrl_04_2_C_0.tiff   -> token[1] = "Ctrl"   -> Tri_Ctrl
      Tri_5ugLPS_04_2_C_0.tiff -> token[1] = "5ugLPS" -> Tri_5ugLPS
    If your naming is different, adjust _infer_treatment_from_stem().

    overlay_ring_interval_microns: ring spacing drawn on the overlay image
    in continuous mode only (drawing every pixel would fill the image).
    """
    import os
    from datetime import datetime

    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import label as ndi_label
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    trapezoid = getattr(np, "trapezoid", None) or np.trapz

    # ---- sampling mode ----
    continuous = step_radius_microns is None or (
        isinstance(step_radius_microns, str)
        and step_radius_microns.strip().lower() in {"continuous", "cont"}
    )

    if not os.path.isdir(input_folder):
        raise FileNotFoundError(f"input_folder not found: {input_folder}")

    if start_radius_microns < 0:
        raise ValueError("start_radius_microns must be >= 0")
    if not continuous:
        if isinstance(step_radius_microns, str):
            raise ValueError("step_radius_microns must be a number, 'continuous', or None")
        if step_radius_microns <= 0:
            raise ValueError("step_radius_microns must be > 0")
    if max_radius_microns <= start_radius_microns:
        raise ValueError("max_radius_microns must be > start_radius_microns")

    def _safe_float_tag(x, decimals=2):
        s = f"{float(x):.{decimals}f}"
        return s.replace(".", "p")

    def _excel_sheet_name(name, max_len=31):
        bad = [":", "\\", "/", "?", "*", "[", "]"]
        for ch in bad:
            name = name.replace(ch, "_")
        return name[:max_len]

    def _infer_treatment_from_stem(stem):
        """
        Infer treatment from the second underscore-delimited token of the stem.
        e.g. Tri_Ctrl_04_2_C_0 -> token[1] = "Ctrl" -> Tri_Ctrl
        """
        tokens = stem.split("_")
        if len(tokens) < 2:
            return "Unknown"

        tok = tokens[1].strip().lower()

        if tok in {"ctrl", "control"}:
            return "Tri_Ctrl"
        if tok in {"5uglps", "lps5ug", "5ug", "5ugl"}:
            return "Tri_5ugLPS"
        if tok in {"500ng", "500lps", "lps500ng"}:
            return "Tri_500ng"
        if tok in {"100ng", "100lps", "lps100ng"}:
            return "Tri_100ng"
        if tok in {"5ng", "5lps", "lps5ng"}:
            return "Tri_5ng"

        return "Unknown"

    step_tag = "continuous" if continuous else f"step_{_safe_float_tag(step_radius_microns)}um"
    out_name = f"{output_folder_name}_startR_{_safe_float_tag(start_radius_microns)}um_{step_tag}"
    output_folder = os.path.join(input_folder, out_name)
    os.makedirs(output_folder, exist_ok=True)

    # find pairs
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(list(c0.keys() & c3.keys()))
    if not keys:
        raise ValueError("No matching _C_0.tiff / _C_3.tiff pairs found in input_folder.")

    # radii
    start_px = int(round(start_radius_microns / scale_microns_per_pixel))
    max_px = int(round(max_radius_microns / scale_microns_per_pixel))
    if continuous:
        step_px = 1
        band_half_width = 0.5   # 1-px-wide band so neighboring radii do not overlap
        overlay_step_px = max(int(round(overlay_ring_interval_microns / scale_microns_per_pixel)), 1)
    else:
        step_px = max(int(round(step_radius_microns / scale_microns_per_pixel)), 1)
        band_half_width = 1.0
        overlay_step_px = step_px
    radii_px = list(range(start_px, max_px + 1, step_px))
    radii_um = [r * scale_microns_per_pixel for r in radii_px]
    overlay_radii_px = list(range(start_px, max_px + 1, overlay_step_px))

    def _li_mask(gray_u8):
        t = float(threshold_li(gray_u8))
        return (gray_u8 > t), t

    def _count_intersections(skel_bool, cx, cy):
        labeled, nfeat = ndi_label(skel_bool.astype(np.uint8))

        rr, cc = np.ogrid[:skel_bool.shape[0], :skel_bool.shape[1]]
        dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)

        counts = []
        for r in radii_px:
            band = (dist >= (r - band_half_width)) & (dist < (r + band_half_width))
            if nfeat == 0:
                counts.append(0)
            else:
                touching = np.unique(labeled[band])
                touching = touching[touching != 0]
                counts.append(int(touching.size))
        return counts

    def _save_overlay(gray_u8, skel_bool, cx, cy, out_png):
        overlay = cv2.cvtColor(gray_u8, cv2.COLOR_GRAY2BGR)
        overlay[skel_bool] = (0, 0, 255)

        rr, cc = np.ogrid[:skel_bool.shape[0], :skel_bool.shape[1]]
        dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)

        for r in overlay_radii_px:
            cv2.circle(overlay, (int(cx), int(cy)), int(r), (0, 255, 0), 1)
            band = (dist >= (r - band_half_width)) & (dist < (r + band_half_width))
            coords = np.argwhere(skel_bool & band)
            for y, x in coords:
                cv2.circle(overlay, (int(x), int(y)), 2, (255, 105, 180), -1)

        cv2.imwrite(out_png, overlay)

    # logging params
    params_file = os.path.join(output_folder, "Parameters.txt")
    with open(params_file, "w") as f:
        f.write("=" * 70 + "\n")
        f.write("SHOLL ANALYSIS - STANDARD SKELETON (C0)\n")
        f.write("=" * 70 + "\n\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Input folder: {input_folder}\n")
        f.write(f"Output folder: {output_folder}\n\n")
        f.write(f"start_radius_microns: {start_radius_microns}\n")
        if continuous:
            f.write(f"step_radius_microns:  continuous (every 1 px = {scale_microns_per_pixel} um)\n")
            f.write(f"overlay_ring_interval_microns: {overlay_ring_interval_microns}\n")
        else:
            f.write(f"step_radius_microns:  {step_radius_microns}\n")
        f.write(f"max_radius_microns:   {max_radius_microns}\n")
        f.write(f"n_radii_sampled:      {len(radii_px)}\n")
        f.write(f"scale_microns_per_pixel: {scale_microns_per_pixel}\n")
        f.write("Center: from _C_3 (calculate_center_of_mass)\n")
        f.write("Mask + skeleton: Li threshold on _C_0 then skeletonize(mask)\n")
        f.write("Mean curves split by Treatment inferred from filename token[1]\n")

    imagelevel_rows = []
    summary_rows = []

    # store curves by treatment for mean curves
    all_counts_by_trt = {"Tri_Ctrl": [], "Tri_5ugLPS": [], "Unknown": []}

    # markers are cluttered with hundreds of points
    marker = None if continuous else "o"

    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])
        output_base = os.path.join(output_folder, os.path.splitext(c0[key])[0])

        # infer treatment from file stem (based on C0 name)
        stem = os.path.splitext(c0[key])[0]
        trt = _infer_treatment_from_stem(stem)

        soma_xy = calculate_center_of_mass(c0_path, c3_path, output_base)
        if soma_xy is None:
            continue
        cx, cy = soma_xy

        c0_img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if c0_img is None:
            print(f"Warning: unable to load {c0_path}, skipping.")
            continue

        mask_c0, li_t_c0 = _li_mask(c0_img)
        skel = skeletonize(mask_c0).astype(bool)
        counts = _count_intersections(skel, cx, cy)

        all_counts_by_trt.setdefault(trt, []).append(counts)

        overlay_png = f"{output_base}_C0_standardSkeleton_overlay.png"
        _save_overlay(c0_img, skel, cx, cy, overlay_png)

        for d_um, c_int in zip(radii_um, counts):
            imagelevel_rows.append(
                {
                    "Key": key,
                    "Treatment": trt,
                    "Image_C0": c0[key],
                    "Image_C3": c3[key],
                    "Li_Threshold_C0": li_t_c0,
                    "Distance_um": float(d_um),
                    "Intersections": int(c_int),
                }
            )

        auc = float(trapezoid(counts, radii_um))
        max_int = int(np.max(counts)) if counts else 0
        peak_idx = int(np.argmax(counts)) if counts else 0
        peak_radius_um = float(radii_um[peak_idx]) if counts else np.nan

        summary_rows.append(
            {
                "Key": key,
                "Treatment": trt,
                "Image_C0": c0[key],
                "Image_C3": c3[key],
                "Li_Threshold_C0": li_t_c0,
                "AUC": auc,
                "Max_Intersections": max_int,
                "Peak_Radius_um": peak_radius_um,
            }
        )

        plt.figure(figsize=(9, 6))
        plt.plot(radii_um, counts, marker=marker, linewidth=2, label="standard_skeletonize")
        plt.title(f"Sholl Curve (standard skeleton) | {key} | {trt}")
        plt.xlabel("Distance from soma (µm)")
        plt.ylabel("Intersections")
        plt.grid(True, alpha=0.3)
        plt.legend()
        curve_png = f"{output_base}_C0_curve.png"
        plt.tight_layout()
        plt.savefig(curve_png, dpi=300, bbox_inches="tight")
        plt.close()

    imagelevel_df = pd.DataFrame(imagelevel_rows)
    summary_df = pd.DataFrame(summary_rows)
    if imagelevel_df.empty:
        raise ValueError("No results generated. Check that images exist and center-of-mass is found.")

    # ---- Mean curves by treatment ----
    mean_rows = []
    plt.figure(figsize=(9, 6))

    for trt, curves in all_counts_by_trt.items():
        if trt == "Unknown":
            continue
        if not curves:
            continue

        mat = np.asarray(curves, dtype=float)
        mean_curve = np.mean(mat, axis=0)
        sem_curve = (
            np.std(mat, axis=0, ddof=1) / np.sqrt(mat.shape[0])
            if mat.shape[0] > 1
            else np.zeros_like(mean_curve)
        )

        for d_um, m, s in zip(radii_um, mean_curve, sem_curve):
            mean_rows.append(
                {
                    "Treatment": trt,
                    "Distance_um": float(d_um),
                    "Mean_Intersections": float(m),
                    "SEM": float(s),
                    "N_Images": int(mat.shape[0]),
                }
            )

        plt.plot(radii_um, mean_curve, marker=marker, linewidth=2, label=f"{trt} (n={mat.shape[0]})")
        plt.fill_between(radii_um, mean_curve - sem_curve, mean_curve + sem_curve, alpha=0.18)

    plt.title("Mean Sholl Curves by Treatment (standard skeleton, C0)")
    plt.xlabel("Distance from soma (µm)")
    plt.ylabel("Mean intersections")
    plt.grid(True, alpha=0.3)
    plt.legend()
    mean_png = os.path.join(output_folder, "Mean_Sholl_Curves_by_Treatment.png")
    plt.tight_layout()
    plt.savefig(mean_png, dpi=300, bbox_inches="tight")
    plt.close()

    mean_df = pd.DataFrame(mean_rows)

    # warn if unknowns exist
    n_unknown = len(all_counts_by_trt.get("Unknown", []))
    if n_unknown > 0:
        print(f"Warning: {n_unknown} image(s) were labeled Treatment=Unknown. Check filename parsing.")

    # wide intersections (include Treatment)
    wide_df = (
        imagelevel_df.pivot_table(
            index=["Key", "Treatment", "Image_C0", "Image_C3", "Li_Threshold_C0"],
            columns="Distance_um",
            values="Intersections",
            aggfunc="first",
        )
        .reset_index()
    )
    wide_df.columns = [col if isinstance(col, str) else f"{float(col):.3f}um" for col in wide_df.columns]

    # excel
    excel_path = os.path.join(output_folder, "Sholl_StandardSkeleton_ByTreatment.xlsx")
    with pd.ExcelWriter(excel_path) as writer:
        imagelevel_df.to_excel(writer, sheet_name="ImageLevel", index=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        mean_df.to_excel(writer, sheet_name="MeanCurves_ByTreatment", index=False)
        wide_df.to_excel(writer, sheet_name=_excel_sheet_name("Intersections_Wide"), index=False)

        # Sheet listing Unknown treatment keys
        if "Unknown" in imagelevel_df["Treatment"].unique():
            unk = imagelevel_df[imagelevel_df["Treatment"] == "Unknown"][
                ["Key", "Image_C0", "Image_C3"]
            ].drop_duplicates()
            unk.to_excel(writer, sheet_name="Unknown_Treatment", index=False)

    print(f"Done. Results in: {output_folder}")
    return {
        "output_folder": output_folder,
        "excel_path": excel_path,
        "imagelevel_df": imagelevel_df,
        "summary_df": summary_df,
        "mean_df": mean_df,
        "wide_df": wide_df,
    }

results_by_step = {}

for step in [2.5, 5, "continuous"]:
    results_by_step[str(step)] = Process_Sholl_StandardSkeleton(
    input_folder='/Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI',
    output_folder_name="261014_Sholl Analysis_Step Size Comparison",
        start_radius_microns=0,
        step_radius_microns=step,
        max_radius_microns=100,
        scale_microns_per_pixel=0.335,
    )
# Process_Sholl_StandardSkeleton(
#     input_folder='/Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI',
#     output_folder_name="0_Sholl Analysis",
#     start_radius_microns=0,
#     step_radius_microns=5,
#     max_radius_microns=100,
#     scale_microns_per_pixel=0.335,
# )

# Continuous sampling
Process_Sholl_StandardSkeleton(
    input_folder='/Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI',
    output_folder_name="261014_Sholl Analysis_Step Size Comparison",
    start_radius_microns=0,
    step_radius_microns="continuous",
    
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
)



Done. Results in: /Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI/261014_Sholl Analysis_Step Size Comparison_startR_0p00um_step_2p50um
Done. Results in: /Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI/261014_Sholl Analysis_Step Size Comparison_startR_0p00um_step_5p00um
Done. Results in: /Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI/261014_Sholl Analysis_Step Size Comparison_startR_0p00um_continuous
Done. Results in: /Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI/261014_Sholl Analysis_Step Size Comparison_startR_0p00um_continuous


{'output_folder': '/Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI/261014_Sholl Analysis_Step Size Comparison_startR_0p00um_continuous',
 'excel_path': '/Users/allisonpickle/Desktop/260501_Morphology Pipeline/Sholl Analysis/All/Isolated_Microglia_DAPI/261014_Sholl Analysis_Step Size Comparison_startR_0p00um_continuous/Sholl_StandardSkeleton_ByTreatment.xlsx',
 'imagelevel_df':                    Key   Treatment                  Image_C0  \
 0      Tri_5ugLPS_01_1  Tri_5ugLPS  Tri_5ugLPS_01_1_C_0.tiff   
 1      Tri_5ugLPS_01_1  Tri_5ugLPS  Tri_5ugLPS_01_1_C_0.tiff   
 2      Tri_5ugLPS_01_1  Tri_5ugLPS  Tri_5ugLPS_01_1_C_0.tiff   
 3      Tri_5ugLPS_01_1  Tri_5ugLPS  Tri_5ugLPS_01_1_C_0.tiff   
 4      Tri_5ugLPS_01_1  Tri_5ugLPS  Tri_5ugLPS_01_1_C_0.tiff   
 ...                ...         ...                       ...   
 80995    Tri_Ctrl_35_4    Tri_Ctrl    Tri_Ctrl_35_4_C_0.tiff   
 80996    Tri_Ctrl_35_4    Tri_Ctrl    Tri_Ctrl_35

Continuous Sampling Sholl comparison code

In [ ]:
def Process_Sholl_StandardSkeleton(
    input_folder,
    start_radius_microns=0,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - Standard Skeleton",
):
    """
    Sholl analysis with:
    - Treatment-specific plots (all 3 sampling methods per treatment)
    - ALL treatments on ONE plot (continuous sampling only)
    - Ordered legend: Ctrl → 5ng → 100ng → 500ng → 5µg (top to bottom)
    - Unique colors per treatment
    - NO timestamp in folder name
    - Saves DAPI centroid visualization images to "DAPI Centroid" subfolder
    - ONLY Tri treatments
    """

    import os
    from datetime import datetime
    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import label as ndi_label
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    # -----------------------------
    # OUTPUT FOLDER (NO TIMESTAMP)
    # -----------------------------
    output_folder = os.path.join(
        input_folder,
        output_folder_name
    )
    os.makedirs(output_folder, exist_ok=True)

    # Create subfolder for DAPI centroid images
    centroid_folder = os.path.join(output_folder, "DAPI Centroid")
    os.makedirs(centroid_folder, exist_ok=True)

    # -----------------------------
    # FILE PAIRS
    # -----------------------------
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(set(c0.keys()) & set(c3.keys()))

    if not keys:
        raise ValueError("No C0/C3 pairs found")

    # -----------------------------
    # TREATMENT ORDER & COLORS (TRI ONLY)
    # -----------------------------
    # Define exact order for Tri treatments
    treatment_order = ["Tri_Ctrl", "Tri_5ng", "Tri_100ng", "Tri_500ng", "Tri_5ug"]
    
    # Color mapping for Tri treatments
    treatment_colors = {
        "Tri_Ctrl": "#3498DB",    # Blue
        "Tri_5ng": "#E67E22",     # Orange
        "Tri_100ng": "#2ECC71",   # Green
        "Tri_500ng": "#DDA0DD",   # Purple
        "Tri_5ug": "#E91E63",     # Pink
    }

    # -----------------------------
    # SAMPLING MODES
    # -----------------------------
    sampling_modes = {
        "2.5um": 2.5,
        "5um": 5.0,
        "continuous": scale_microns_per_pixel,
    }

    # Color mapping for sampling methods
    sampling_colors = {
        "2.5um": "#E74C3C",      # Red
        "5um": "#3498DB",        # Blue
        "continuous": "#2ECC71"  # Green
    }

    # -----------------------------
    # HELPERS
    # -----------------------------
    def _li_mask(img):
        t = threshold_li(img)
        return img > t, t

    def _count_intersections(skel, cx, cy, radii_px):
        labeled, _ = ndi_label(skel.astype(np.uint8))
        rr, cc = np.ogrid[:skel.shape[0], :skel.shape[1]]
        dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)

        counts = []
        for r in radii_px:
            band = (dist >= (r - 1)) & (dist <= (r + 1))
            touching = np.unique(labeled[band])
            touching = touching[touching != 0]
            counts.append(len(touching))
        return counts

    def _save_centroid_image(c3_path, cx, cy, stem, output_folder):
        """
        Load C_3 (DAPI) image, draw centroid marker, and save to output folder.
        """
        img = cv2.imread(c3_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return
        
        # Convert to RGB for colored marker
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        # Draw centroid: red cross + circle
        cv2.drawMarker(img_rgb, (int(cx), int(cy)), 
                      color=(255, 0, 0), markerType=cv2.MARKER_CROSS, 
                      markerSize=20, thickness=2)
        cv2.circle(img_rgb, (int(cx), int(cy)), 
                  radius=10, color=(0, 255, 0), thickness=2)
        
        # Save to DAPI Centroid folder
        out_filename = f"{stem}_centroid.png"
        out_path = os.path.join(output_folder, out_filename)
        cv2.imwrite(out_path, img_rgb)
        
        return out_path

    # -----------------------------
    # STORAGE
    # -----------------------------
    image_rows = []
    summary_rows = []
    all_curves = {}
    all_radii_um = {}

    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    print(f"\nProcessing {len(keys)} image pairs...")
    print("Looking for Tri treatments only (Tri_Ctrl, Tri_5ng, Tri_100ng, Tri_500ng, Tri_5ug)\n")
    
    for key in keys:

        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])

        stem = os.path.splitext(c0[key])[0]
        tokens = stem.split("_")
        
        # Treatment is FIRST TWO elements joined by underscore
        # e.g., "Tri_Ctrl_001_C_0" -> tokens[0]="Tri", tokens[1]="Ctrl" -> "Tri_Ctrl"
        if len(tokens) >= 2:
            trt = f"{tokens[0]}_{tokens[1]}"
        else:
            trt = "Unknown"

        # FILTER: Only process Tri treatments
        if not trt.startswith("Tri_"):
            print(f"  ⊗ Skipping {stem} (treatment: {trt}) - not a Tri treatment")
            continue

        print(f"  → Processing {stem} (treatment: {trt})")

        soma_xy = calculate_center_of_mass(c0_path, c3_path, stem)
        if soma_xy is None:
            print(f"    ✗ Failed to find center of mass")
            continue
        cx, cy = soma_xy

        # Save centroid visualization on C_3 image
        centroid_img_path = _save_centroid_image(c3_path, cx, cy, stem, centroid_folder)
        if centroid_img_path:
            print(f"    ✓ Saved centroid: {os.path.basename(centroid_img_path)}")

        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"    ✗ Failed to load C0 image")
            continue

        mask, li_t = _li_mask(img)
        skel = skeletonize(mask)

        for samp_name, step_um in sampling_modes.items():

            # -----------------------------
            # DEFINE STEP SIZE
            # -----------------------------
            if samp_name == "continuous":
                step_um_use = scale_microns_per_pixel  # 1 pixel step
            else:
                step_um_use = float(step_um)

            # -----------------------------
            # BUILD RADII IN µm SPACE (SAFE GRID)
            # -----------------------------
            n_steps = int(np.floor((max_radius_microns - start_radius_microns) / step_um_use)) + 1

            radii_um = start_radius_microns + np.arange(n_steps) * step_um_use

            # enforce strict cutoff
            radii_um = radii_um[radii_um <= max_radius_microns]

            # -----------------------------
            # CONVERT TO PIXELS
            # -----------------------------
            radii_px = np.round(radii_um / scale_microns_per_pixel).astype(int)

            # remove duplicates (important when step is small vs pixel size)
            radii_px = np.unique(radii_px)

            # safety: ensure at least 2 radii
            if len(radii_px) < 2:
                continue

            # Recalculate actual radii_um after deduplication
            radii_um_actual = radii_px * scale_microns_per_pixel

            # -----------------------------
            # RUN SHOLL
            # -----------------------------
            counts = _count_intersections(skel, cx, cy, radii_px)

            all_curves.setdefault((trt, samp_name), []).append(counts)
            all_radii_um.setdefault((trt, samp_name), []).append(radii_um_actual)

            auc = np.trapz(counts, radii_um_actual)

            image_rows.append({
                "Key": key,
                "Treatment": trt,
                "Sampling": samp_name,
                "AUC": auc,
            })

            summary_rows.append({
                "Key": key,
                "Treatment": trt,
                "Sampling": samp_name,
                "AUC": auc,
            })

    # -----------------------------
    # CHECK IF ANY TRI TREATMENTS FOUND
    # -----------------------------
    if not image_rows:
        raise ValueError(
            "No Tri treatments found in the data!\n"
            "Expected treatments: Tri_Ctrl, Tri_5ng, Tri_100ng, Tri_500ng, Tri_5ug\n"
            f"Check your file naming convention. Found {len(keys)} image pairs, but none matched Tri_ prefix."
        )

    # -----------------------------
    # DATAFRAMES
    # -----------------------------
    df_img = pd.DataFrame(image_rows)
    df_sum = pd.DataFrame(summary_rows)

    # Get only Tri treatments that exist in data, in the correct order
    treatments_in_data = set(df_img["Treatment"])
    ordered_treatments = [t for t in treatment_order if t in treatments_in_data]

    print(f"\n✓ Found {len(ordered_treatments)} Tri treatments: {', '.join(ordered_treatments)}")

    # -----------------------------
    # PLOT 1: PER TREATMENT - ALL SAMPLING METHODS
    # -----------------------------
    mean_rows = []

    for trt in ordered_treatments:

        plt.figure(figsize=(10, 6))

        for samp in sampling_modes.keys():

            curves = all_curves.get((trt, samp), None)
            radii_lists = all_radii_um.get((trt, samp), None)
            
            if not curves or not radii_lists:
                continue

            # Find the minimum length across all curves for this treatment/sampling combo
            min_len = min(len(c) for c in curves)
            
            # Truncate all curves and radii to the same length
            curves_trimmed = [c[:min_len] for c in curves]
            radii_trimmed = [r[:min_len] for r in radii_lists]

            mat = np.array(curves_trimmed, dtype=float)
            radii_mat = np.array(radii_trimmed, dtype=float)
            
            mean_counts = mat.mean(axis=0)
            sem_counts = mat.std(axis=0) / np.sqrt(mat.shape[0])
            mean_radii = radii_mat.mean(axis=0)  # Use actual µm values

            color = sampling_colors[samp]

            plt.plot(mean_radii, mean_counts, 
                    label=f"{samp} (n={mat.shape[0]})", 
                    color=color, linewidth=2)
            plt.fill_between(mean_radii, 
                            mean_counts - sem_counts, 
                            mean_counts + sem_counts, 
                            alpha=0.2, color=color)

            for i, (r, m) in enumerate(zip(mean_radii, mean_counts)):
                mean_rows.append({
                    "Treatment": trt,
                    "Sampling": samp,
                    "Radius_um": float(r),
                    "Mean_Intersections": float(m),
                })

        plt.title(f"Sholl Analysis — {trt}", fontsize=14, fontweight='bold')
        plt.xlabel("Radius (µm)", fontsize=12)
        plt.ylabel("Mean Intersections", fontsize=12)
        plt.grid(alpha=0.3, linestyle='--')
        plt.legend(fontsize=10, framealpha=0.9)

        out_path = os.path.join(output_folder, f"Sholl_Combined_{trt}.png")
        plt.tight_layout()
        plt.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close()

        print(f"✓ Saved plot: {os.path.basename(out_path)}")

    # -----------------------------
    # PLOT 2: ALL TRI TREATMENTS - CONTINUOUS SAMPLING ONLY (ORDERED)
    # -----------------------------
    print("\n" + "="*60)
    print("Creating all-treatments plot (continuous sampling)...")
    
    plt.figure(figsize=(12, 7))

    for trt in ordered_treatments:
        
        curves = all_curves.get((trt, "continuous"), None)
        radii_lists = all_radii_um.get((trt, "continuous"), None)
        
        if not curves or not radii_lists:
            continue

        # Find the minimum length across all curves for this treatment
        min_len = min(len(c) for c in curves)
        
        # Truncate all curves and radii to the same length
        curves_trimmed = [c[:min_len] for c in curves]
        radii_trimmed = [r[:min_len] for r in radii_lists]

        mat = np.array(curves_trimmed, dtype=float)
        radii_mat = np.array(radii_trimmed, dtype=float)
        
        mean_counts = mat.mean(axis=0)
        sem_counts = mat.std(axis=0) / np.sqrt(mat.shape[0])
        mean_radii = radii_mat.mean(axis=0)

        # Use predefined color
        color = treatment_colors[trt]

        plt.plot(mean_radii, mean_counts, 
                label=f"{trt} (n={mat.shape[0]})", 
                color=color, linewidth=2.5, alpha=0.8)
        plt.fill_between(mean_radii, 
                        mean_counts - sem_counts, 
                        mean_counts + sem_counts, 
                        alpha=0.15, color=color)

    plt.title("Sholl Analysis — All Treatments (Continuous Sampling)", 
             fontsize=16, fontweight='bold')
    plt.xlabel("Radius (µm)", fontsize=13)
    plt.ylabel("Mean Intersections", fontsize=13)
    plt.grid(alpha=0.3, linestyle='--')
    
    # Legend with ordered treatments (Ctrl → 5ng → 100ng → 500ng → 5µg)
    plt.legend(fontsize=9, framealpha=0.95, loc='best')

    out_path_all = os.path.join(output_folder, "Sholl_AllTreatments_Continuous.png")
    plt.tight_layout()
    plt.savefig(out_path_all, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Saved all-treatments plot: {os.path.basename(out_path_all)}")

    # -----------------------------
    # EXPORT
    # -----------------------------
    excel_path = os.path.join(output_folder, "Sholl_ALL_Sampling.xlsx")

    with pd.ExcelWriter(excel_path) as writer:
        df_img.to_excel(writer, sheet_name="ImageLevel", index=False)
        df_sum.to_excel(writer, sheet_name="Summary", index=False)
        pd.DataFrame(mean_rows).to_excel(writer, sheet_name="MeanCurves", index=False)

    print(f"\n{'='*60}")
    print(f"DONE")
    print(f"Output folder: {output_folder}")
    print(f"Centroid images: {centroid_folder}")
    print(f"Excel: {excel_path}")
    print(f"{'='*60}\n")

    return {
        "output_folder": output_folder,
        "centroid_folder": centroid_folder,
        "excel_path": excel_path,
        "image_df": df_img,
        "summary_df": df_sum,
    }

In [ ]:
def Process_Sholl_StandardSkeleton_BioReps(
    input_folder,
    start_radius_microns=0,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - Standard Skeleton",
):
    """
    Sholl analysis with:
    - Handles 3 biological replicates (1_, 2_, 3_)
    - Treatment-specific plots (all 3 sampling methods per treatment)
    - ALL treatments on ONE plot (continuous sampling only)
    - Separate plots per time point (e.g., 96hr, 48hr)
    - Ordered legend: Ctrl → 5ng → 100ng → 500ng → 5µg (top to bottom)
    - Unique colors per treatment
    - NO timestamp in folder name
    - Saves DAPI centroid visualization images to "DAPI Centroid" subfolder
    - ONLY Tri treatments
    """

    import os
    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.ndimage import label as ndi_label
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    # -----------------------------
    # OUTPUT FOLDER (NO TIMESTAMP)
    # -----------------------------
    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    # Create subfolder for DAPI centroid images
    centroid_folder = os.path.join(output_folder, "DAPI Centroid")
    os.makedirs(centroid_folder, exist_ok=True)

    # -----------------------------
    # FILE PAIRS
    # -----------------------------
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(set(c0.keys()) & set(c3.keys()))

    if not keys:
        raise ValueError("No C0/C3 pairs found")

    # -----------------------------
    # TREATMENT ORDER & COLORS (TRI ONLY)
    # -----------------------------
    # Define exact order for Tri treatments
    treatment_order = ["Tri_Ctrl", "Tri_5ng", "Tri_100ng", "Tri_500ng", "Tri_5ug"]
    
    # Color mapping for Tri treatments
    treatment_colors = {
        "Tri_Ctrl": "#3498DB",    # Blue
        "Tri_5ng": "#E67E22",     # Orange
        "Tri_100ng": "#2ECC71",   # Green
        "Tri_500ng": "#DDA0DD",   # Purple
        "Tri_5ug": "#E91E63",     # Pink
    }

    # -----------------------------
    # SAMPLING MODES
    # -----------------------------
    sampling_modes = {
        "2.5um": 2.5,
        "5um": 5.0,
        "continuous": scale_microns_per_pixel,
    }

    # -----------------------------
    # STORAGE
    # -----------------------------
    image_rows = []
    summary_rows = []
    all_curves = {}
    all_radii_um = {}

    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    print(f"\nProcessing {len(keys)} image pairs...")
    print("Looking for Tri treatments only (Tri_Ctrl, Tri_5ng, Tri_100ng, Tri_500ng, Tri_5ug)\n")
    
    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])

        stem = os.path.splitext(c0[key])[0]
        tokens = stem.split("_")
        
        # Biological replicate is the first element (e.g., 1_, 2_, 3_)
        if len(tokens) < 3:
            print(f"  ⊗ Skipping {stem} - invalid filename format")
            continue
        bio_rep = tokens[0]
        time_point = tokens[1]
        treatment = f"{tokens[2]}_{tokens[3]}"

        # FILTER: Only process Tri treatments
        if not treatment.startswith("Tri_"):
            print(f"  ⊗ Skipping {stem} (treatment: {treatment}) - not a Tri treatment")
            continue

        print(f"  → Processing {stem} (treatment: {treatment}, time point: {time_point}, bio rep: {bio_rep})")

        # Calculate centroid
        soma_xy = calculate_center_of_mass(c0_path, c3_path, stem)
        if soma_xy is None:
            print(f"    ✗ Failed to find center of mass")
            continue
        cx, cy = soma_xy

        # Save centroid visualization on C_3 image
        centroid_img_path = _save_centroid_image(c3_path, cx, cy, stem, centroid_folder)
        if centroid_img_path:
            print(f"    ✓ Saved centroid: {os.path.basename(centroid_img_path)}")

        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"    ✗ Failed to load C0 image")
            continue

        mask, li_t = _li_mask(img)
        skel = skeletonize(mask)

        for samp_name, step_um in sampling_modes.items():
            # Define step size
            step_um_use = scale_microns_per_pixel if samp_name == "continuous" else float(step_um)

            # Build radii in µm space
            n_steps = int(np.floor((max_radius_microns - start_radius_microns) / step_um_use)) + 1
            radii_um = start_radius_microns + np.arange(n_steps) * step_um_use
            radii_um = radii_um[radii_um <= max_radius_microns]

            # Convert to pixels
            radii_px = np.round(radii_um / scale_microns_per_pixel).astype(int)
            radii_px = np.unique(radii_px)

            if len(radii_px) < 2:
                continue

            radii_um_actual = radii_px * scale_microns_per_pixel
            counts = _count_intersections(skel, cx, cy, radii_px)

            all_curves.setdefault((treatment, time_point, samp_name, bio_rep), []).append(counts)
            all_radii_um.setdefault((treatment, time_point, samp_name, bio_rep), []).append(radii_um_actual)

    # -----------------------------
    # PLOT PER TIME POINT
    # -----------------------------
    for time_point in set(k[1] for k in all_curves.keys()):
        for treatment in treatment_order:
            plt.figure(figsize=(10, 6))

            for samp_name in sampling_modes.keys():
                for bio_rep in ["1", "2", "3"]:
                    curves = all_curves.get((treatment, time_point, samp_name, bio_rep), None)
                    radii_lists = all_radii_um.get((treatment, time_point, samp_name, bio_rep), None)

                    if not curves or not radii_lists:
                        continue

                    mat = np.array(curves, dtype=float)
                    radii_mat = np.array(radii_lists, dtype=float)

                    mean_counts = mat.mean(axis=0)
                    sem_counts = mat.std(axis=0) / np.sqrt(mat.shape[0])
                    mean_radii = radii_mat.mean(axis=0)

                    plt.plot(mean_radii, mean_counts, label=f"{samp_name} (Rep {bio_rep})", linewidth=2)
                    plt.fill_between(mean_radii, mean_counts - sem_counts, mean_counts + sem_counts, alpha=0.2)

            plt.title(f"Sholl Analysis — {treatment} ({time_point})", fontsize=14, fontweight='bold')
            plt.xlabel("Radius (µm)", fontsize=12)
            plt.ylabel("Mean Intersections", fontsize=12)
            plt.legend(fontsize=10)
            plt.grid(alpha=0.3, linestyle='--')

            out_path = os.path.join(output_folder, f"Sholl_{treatment}_{time_point}.png")
            plt.tight_layout()
            plt.savefig(out_path, dpi=300)
            plt.close()

            print(f"✓ Saved plot: {os.path.basename(out_path)}")



In [ ]:
# Define the input folder containing your image files
input_folder = '/Users/allisonpickle/Desktop/LPS/3 BioReps Sholl/All_Bio_Reps/Isolated_Microglia_DAPI'

# Call the function
Process_Sholl_StandardSkeleton_BioReps(
    input_folder=input_folder,
    start_radius_microns=0,  # Starting radius in microns
    max_radius_microns=100,  # Maximum radius in microns
    scale_microns_per_pixel=0.335,  # Scale: microns per pixel
    output_folder_name="Sholl Analysis - Standard Skeleton"  # Output folder name
)


Processing 2883 image pairs...
Looking for Tri treatments only (Tri_Ctrl, Tri_5ng, Tri_100ng, Tri_500ng, Tri_5ug)

  ⊗ Skipping 1_24hr_100ng_01_1_C_0 (treatment: 100ng_01) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_01_2_C_0 (treatment: 100ng_01) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_01_3_C_0 (treatment: 100ng_01) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_01_4_C_0 (treatment: 100ng_01) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_01_5_C_0 (treatment: 100ng_01) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_02_1_C_0 (treatment: 100ng_02) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_02_2_C_0 (treatment: 100ng_02) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_02_3_C_0 (treatment: 100ng_02) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_03_1_C_0 (treatment: 100ng_03) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_03_2_C_0 (treatment: 100ng_03) - not a Tri treatment
  ⊗ Skipping 1_24hr_100ng_03_3_C_0 (treatment: 100ng_03) - not a Tri treatment
  ⊗ Skipping 1_

NameError: name 'calculate_center_of_mass' is not defined

In [ ]:
Process_Sholl_StandardSkeleton(
    input_folder = '/Users/allisonpickle/Desktop/LPS/3 BioReps Sholl/All_Bio_Reps/Isolated_Microglia_DAPI',
    start_radius_microns = 0,
    max_radius_microns=50,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis",
)

ValueError: No C0/C3 pairs found

UPDATE 10/2/26 SHOLL ANALYSIS vv

In [ ]:
from scipy.ndimage import center_of_mass
import cv2

def calculate_center_of_mass(c0_path, c3_path, stem):
    """
    Calculate the center of mass (centroid) for the given image pair.
    
    Parameters:
    - c0_path: str, path to the C_0 image file.
    - c3_path: str, path to the C_3 image file.
    - stem: str, base name of the image file (for debugging/logging purposes).
    
    Returns:
    - (cx, cy): tuple, coordinates of the centroid (center of mass).
    - None: if the calculation fails.
    """
    try:
        # Load the C_0 image (used for skeletonization)
        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"    ✗ Failed to load C_0 image for {stem}")
            return None

        # Threshold the image to create a binary mask
        _, binary_mask = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # Calculate the center of mass using the binary mask
        cy, cx = center_of_mass(binary_mask)

        # Return the coordinates as integers
        return int(cx), int(cy)

    except Exception as e:
        print(f"    ✗ Error calculating center of mass for {stem}: {e}")
        return None
    
import cv2

def _save_centroid_image(c3_path, cx, cy, stem, output_folder):
    """
    Load C_3 (DAPI) image, draw centroid marker, and save to output folder.

    Parameters:
    - c3_path: str, path to the C_3 image file.
    - cx, cy: int, coordinates of the centroid.
    - stem: str, base name of the image file (for naming the output file).
    - output_folder: str, folder where the output image will be saved.

    Returns:
    - out_path: str, path to the saved image.
    - None: if the image cannot be loaded or saved.
    """
    try:
        # Load the C_3 image (DAPI channel)
        img = cv2.imread(c3_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"    ✗ Failed to load C_3 image for {stem}")
            return None

        # Convert to RGB for colored marker
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        # Draw centroid: red cross + green circle
        cv2.drawMarker(img_rgb, (int(cx), int(cy)), 
                       color=(255, 0, 0), markerType=cv2.MARKER_CROSS, 
                       markerSize=20, thickness=2)
        cv2.circle(img_rgb, (int(cx), int(cy)), 
                   radius=10, color=(0, 255, 0), thickness=2)

        # Save to the output folder
        out_filename = f"{stem}_centroid.png"
        out_path = os.path.join(output_folder, out_filename)
        cv2.imwrite(out_path, img_rgb)

        return out_path

    except Exception as e:
        print(f"    ✗ Error saving centroid image for {stem}: {e}")
        return None
    
from scipy.ndimage import label as ndi_label

def _count_intersections(skel, cx, cy, radii_px):
    """
    Count the number of intersections between the skeletonized image and concentric circles.

    Parameters:
    - skel: np.ndarray, the skeletonized binary image.
    - cx, cy: int, coordinates of the centroid.
    - radii_px: list or np.ndarray, radii of the concentric circles in pixels.

    Returns:
    - counts: list, number of intersections for each radius.
    """
    # Label connected components in the skeleton
    labeled, _ = ndi_label(skel.astype(np.uint8))

    # Create a grid of coordinates
    rr, cc = np.ogrid[:skel.shape[0], :skel.shape[1]]
    dist = np.sqrt((rr - cy) ** 2 + (cc - cx) ** 2)  # Distance from centroid

    counts = []
    for r in radii_px:
        # Create a band around the current radius
        band = (dist >= (r - 1)) & (dist <= (r + 1))
        # Count unique labeled components that intersect the band
        touching = np.unique(labeled[band])
        touching = touching[touching != 0]  # Exclude background (label 0)
        counts.append(len(touching))
    return counts

In [ ]:
# For all concentrations within a time point
def Process_Sholl_ContinuousSampling_BioReps(


    input_folder,
    start_radius_microns=0,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - Continuous Sampling",
    y_max=None,   # Upper limit of the y axis (None = autoscale)
    y_min=0,      # Lower limit of the y axis (None = autoscale)
):
    """
    Sholl analysis with:
    - Biological replicates parsed from the first filename element (1, 2, 3)
    - One plot per time point (second filename element) with all concentrations
    - Replicates pooled and averaged per treatment/time point
    - Continuous sampling only
    - Optional fixed y-axis range (y_min, y_max)
    - DAPI centroid images saved to "DAPI Centroid" subfolder
    """

    import os
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    # -----------------------------
    # OUTPUT FOLDERS
    # -----------------------------
    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    centroid_folder = os.path.join(output_folder, "DAPI Centroid")
    os.makedirs(centroid_folder, exist_ok=True)

    # -----------------------------
    # FILE PAIRS
    # -----------------------------
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(set(c0.keys()) & set(c3.keys()))

    if not keys:
        raise ValueError("No C0/C3 pairs found")

    # -----------------------------
    # TREATMENT ORDER & COLORS
    # -----------------------------
    treatment_order = ["Ctrl", "5ng", "100ng", "500ng", "5ug"]
    treatment_colors = {
        "Ctrl": "#3498DB",    # Blue
        "5ng": "#E67E22",     # Orange
        "100ng": "#2ECC71",   # Green
        "500ng": "#DDA0DD",   # Purple
        "5ug": "#E91E63",     # Pink
    }

    # -----------------------------
    # STORAGE
    # -----------------------------
    all_curves = {}
    all_radii_um = {}

    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    print(f"\nProcessing {len(keys)} image pairs...\n")

    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])

        stem = os.path.splitext(c0[key])[0]
        tokens = stem.split("_")

        if len(tokens) < 5:
            print(f"  ⊗ Skipping {stem} - invalid filename format")
            continue

        bio_rep = tokens[0]      # 1, 2, 3
        time_point = tokens[1]   # 6hr, 24hr, 48hr, ...

        # Normalize "Tri_Ctrl" -> "Ctrl"; otherwise use the single token (e.g., "5ng")
        if tokens[2] == "Tri" and tokens[3] == "Ctrl":
            treatment = "Ctrl"
        else:
            treatment = tokens[2]

        print(f"  → Processing {stem} (treatment: {treatment}, time point: {time_point}, bio rep: {bio_rep})")

        # Centroid
        soma_xy = calculate_center_of_mass(c0_path, c3_path, stem)
        if soma_xy is None:
            print("    ✗ Failed to find center of mass")
            continue
        cx, cy = soma_xy

        centroid_img_path = _save_centroid_image(c3_path, cx, cy, stem, centroid_folder)
        if centroid_img_path:
            print(f"    ✓ Saved centroid: {os.path.basename(centroid_img_path)}")

        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print("    ✗ Failed to load C0 image")
            continue

        mask = img > threshold_li(img)
        skel = skeletonize(mask)

        # Continuous sampling (1 pixel step)
        step_um_use = scale_microns_per_pixel
        n_steps = int(np.floor((max_radius_microns - start_radius_microns) / step_um_use)) + 1
        radii_um = start_radius_microns + np.arange(n_steps) * step_um_use
        radii_um = radii_um[radii_um <= max_radius_microns]

        radii_px = np.unique(np.round(radii_um / scale_microns_per_pixel).astype(int))
        if len(radii_px) < 2:
            continue

        radii_um_actual = radii_px * scale_microns_per_pixel
        counts = _count_intersections(skel, cx, cy, radii_px)

        all_curves.setdefault((treatment, time_point), []).append(counts)
        all_radii_um.setdefault((treatment, time_point), []).append(radii_um_actual)

    # Debug: confirm grouped names match treatment_order
    print("\nGrouped keys:", sorted(all_curves.keys()))

    # -----------------------------
    # PLOT PER TIME POINT
    # -----------------------------
    for time_point in sorted(set(k[1] for k in all_curves.keys())):
        plt.figure(figsize=(12, 8))

        for treatment in treatment_order:
            curves = all_curves.get((treatment, time_point))
            radii_lists = all_radii_um.get((treatment, time_point))

            if not curves or not radii_lists:
                continue

            mat = np.array(curves, dtype=float)
            radii_mat = np.array(radii_lists, dtype=float)

            mean_counts = mat.mean(axis=0)
            sem_counts = mat.std(axis=0) / np.sqrt(mat.shape[0])
            mean_radii = radii_mat.mean(axis=0)

            color = treatment_colors.get(treatment, "#000000")
            plt.plot(mean_radii, mean_counts, label=f"{treatment} (n={mat.shape[0]})",
                     color=color, linewidth=2)
            plt.fill_between(mean_radii, mean_counts - sem_counts, mean_counts + sem_counts,
                             alpha=0.2, color=color)

        plt.title(f"Sholl Analysis — {time_point}", fontsize=16, fontweight="bold")
        plt.xlabel("Radius (µm)", fontsize=14)
        plt.ylabel("Mean Intersections", fontsize=14)

        # Fixed y-axis range (applied to every time point plot when y_max is set)
        if y_max is not None:
            plt.ylim(y_min, y_max)

        plt.legend(fontsize=12, framealpha=0.9)
        plt.grid(alpha=0.3, linestyle="--")

        out_path = os.path.join(output_folder, f"Sholl_AllConcentrations_{time_point}.png")
        plt.tight_layout()
        plt.savefig(out_path, dpi=300)
        plt.close()

        print(f"✓ Saved plot: {os.path.basename(out_path)}")

In [ ]:
Process_Sholl_ContinuousSampling_BioReps(
    input_folder="/Users/allisonpickle/Desktop/LPS/3 BioReps Analysis/0_All_Bio_Reps/Sholl Analysis/Microglia_DAPI_Images",
    start_radius_microns=0,
    max_radius_microns=50,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - Compact y axis",
    y_max=18,
)


Processing 3099 image pairs...

  → Processing 1_24hr_100ng_01_1_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_1_C_0_centroid.png
  → Processing 1_24hr_100ng_01_2_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_2_C_0_centroid.png
  → Processing 1_24hr_100ng_01_3_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_3_C_0_centroid.png
  → Processing 1_24hr_100ng_01_4_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_4_C_0_centroid.png
  → Processing 1_24hr_100ng_01_5_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_5_C_0_centroid.png
  → Processing 1_24hr_100ng_02_1_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_02_1_C_0_centroid.png
  → Processing 1_24hr_100ng_02_2_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1

In [ ]:
# Intersection plot by LPS concentration
def Process_Sholl_ContinuousSampling_BioReps_ByConcentration(
    input_folder,
    start_radius_microns=0,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - By Concentration",
    y_max=None,   # Upper limit of the y axis (None = autoscale)
    y_min=0,      # Lower limit of the y axis (None = autoscale)
):
    """
    Sholl analysis with:
    - Biological replicates parsed from the first filename element (1, 2, 3)
    - ONE PLOT PER TREATMENT/CONCENTRATION, with each trace line = a time point
    - Replicates pooled and averaged per treatment/time point
    - Continuous sampling only
    - Optional fixed y-axis range (y_min, y_max)
    - DAPI centroid images saved to "DAPI Centroid" subfolder
    """

    import os
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    # -----------------------------
    # OUTPUT FOLDERS
    # -----------------------------
    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    centroid_folder = os.path.join(output_folder, "DAPI Centroid")
    os.makedirs(centroid_folder, exist_ok=True)

    # -----------------------------
    # FILE PAIRS
    # -----------------------------
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(set(c0.keys()) & set(c3.keys()))

    if not keys:
        raise ValueError("No C0/C3 pairs found")

    # -----------------------------
    # TREATMENT ORDER & TIME POINT ORDER/COLORS
    # -----------------------------
    treatment_order = ["Ctrl", "5ng", "100ng", "500ng", "5ug"]

    time_point_order = ["6hr", "24hr", "48hr", "72hr", "96hr"]
    time_point_colors = {
        "6hr":  "#3498DB",   # Blue
        "24hr": "#E67E22",   # Orange
        "48hr": "#2ECC71",   # Green
        "72hr": "#DDA0DD",   # Purple
        "96hr": "#E91E63",   # Pink
    }

    # -----------------------------
    # STORAGE
    # -----------------------------
    all_curves = {}
    all_radii_um = {}

    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    print(f"\nProcessing {len(keys)} image pairs...\n")

    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])

        stem = os.path.splitext(c0[key])[0]
        tokens = stem.split("_")

        if len(tokens) < 5:
            print(f"  ⊗ Skipping {stem} - invalid filename format")
            continue

        bio_rep = tokens[0]      # 1, 2, 3
        time_point = tokens[1]   # 6hr, 24hr, 48hr, ...

        # Normalize "Tri_Ctrl" -> "Ctrl"; otherwise use the single token (e.g., "5ng")
        if tokens[2] == "Tri" and tokens[3] == "Ctrl":
            treatment = "Ctrl"
        else:
            treatment = tokens[2]

        print(f"  → Processing {stem} (treatment: {treatment}, time point: {time_point}, bio rep: {bio_rep})")

        # Centroid
        soma_xy = calculate_center_of_mass(c0_path, c3_path, stem)
        if soma_xy is None:
            print("    ✗ Failed to find center of mass")
            continue
        cx, cy = soma_xy

        centroid_img_path = _save_centroid_image(c3_path, cx, cy, stem, centroid_folder)
        if centroid_img_path:
            print(f"    ✓ Saved centroid: {os.path.basename(centroid_img_path)}")

        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print("    ✗ Failed to load C0 image")
            continue

        mask = img > threshold_li(img)
        skel = skeletonize(mask)

        # Continuous sampling (1 pixel step)
        step_um_use = scale_microns_per_pixel
        n_steps = int(np.floor((max_radius_microns - start_radius_microns) / step_um_use)) + 1
        radii_um = start_radius_microns + np.arange(n_steps) * step_um_use
        radii_um = radii_um[radii_um <= max_radius_microns]

        radii_px = np.unique(np.round(radii_um / scale_microns_per_pixel).astype(int))
        if len(radii_px) < 2:
            continue

        radii_um_actual = radii_px * scale_microns_per_pixel
        counts = _count_intersections(skel, cx, cy, radii_px)

        all_curves.setdefault((treatment, time_point), []).append(counts)
        all_radii_um.setdefault((treatment, time_point), []).append(radii_um_actual)

    # Debug: confirm grouped names match expected order
    print("\nGrouped keys:", sorted(all_curves.keys()))

    # -----------------------------
    # PLOT PER TREATMENT/CONCENTRATION
    # (each trace line = a time point)
    # -----------------------------
    for treatment in treatment_order:
        # Skip treatments with no data at all
        available_times = [k[1] for k in all_curves.keys() if k[0] == treatment]
        if not available_times:
            continue

        plt.figure(figsize=(12, 8))

        # Use canonical order where possible; fall back to any extra time labels found
        ordered_times = [t for t in time_point_order if t in available_times]
        extra_times = sorted(set(available_times) - set(ordered_times))
        ordered_times += extra_times

        for time_point in ordered_times:
            curves = all_curves.get((treatment, time_point))
            radii_lists = all_radii_um.get((treatment, time_point))

            if not curves or not radii_lists:
                continue

            mat = np.array(curves, dtype=float)
            radii_mat = np.array(radii_lists, dtype=float)

            mean_counts = mat.mean(axis=0)
            sem_counts = mat.std(axis=0) / np.sqrt(mat.shape[0])
            mean_radii = radii_mat.mean(axis=0)

            color = time_point_colors.get(time_point, "#000000")
            plt.plot(mean_radii, mean_counts, label=f"{time_point} (n={mat.shape[0]})",
                     color=color, linewidth=2)
            plt.fill_between(mean_radii, mean_counts - sem_counts, mean_counts + sem_counts,
                             alpha=0.2, color=color)

        plt.title(f"Sholl Analysis — {treatment}", fontsize=16, fontweight="bold")
        plt.xlabel("Radius (µm)", fontsize=14)
        plt.ylabel("Mean Intersections", fontsize=14)

        # Fixed y-axis range (applied to every treatment plot when y_max is set)
        if y_max is not None:
            plt.ylim(y_min, y_max)

        plt.legend(fontsize=12, framealpha=0.9, title="Time Point")
        plt.grid(alpha=0.3, linestyle="--")

        out_path = os.path.join(output_folder, f"Sholl_AllTimePoints_{treatment}.png")
        plt.tight_layout()
        plt.savefig(out_path, dpi=300)
        plt.close()

        print(f"✓ Saved plot: {os.path.basename(out_path)}")

In [ ]:
# Example: generate one plot per concentration, each showing all time points as separate trace lines
Process_Sholl_ContinuousSampling_BioReps_ByConcentration(
    input_folder='/Users/allisonpickle/Desktop/Super LPS/3 BioReps Analysis/0_All_Bio_Reps/Sholl Analysis/Microglia_DAPI_Images',
    start_radius_microns=0,
    max_radius_microns=50,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - By Concentration",
    y_max=18,     # e.g., fix y-axis to 0-20 intersections across all plots
    y_min=0,
)


Processing 3099 image pairs...

  → Processing 1_24hr_100ng_01_1_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_1_C_0_centroid.png
  → Processing 1_24hr_100ng_01_2_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_2_C_0_centroid.png
  → Processing 1_24hr_100ng_01_3_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_3_C_0_centroid.png
  → Processing 1_24hr_100ng_01_4_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_4_C_0_centroid.png
  → Processing 1_24hr_100ng_01_5_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_01_5_C_0_centroid.png
  → Processing 1_24hr_100ng_02_1_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1_24hr_100ng_02_1_C_0_centroid.png
  → Processing 1_24hr_100ng_02_2_C_0 (treatment: 100ng, time point: 24hr, bio rep: 1)
    ✓ Saved centroid: 1

In [ ]:
def Calculate_Sholl_AUC(
    input_folder,
    start_radius_microns=0,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - AUC",
    excel_name="Sholl_AUC_Results.xlsx",
):
    """
    Calculates Sholl AUC (continuous sampling) for each time point and concentration.
    Excel sheets:
    - Per_Image: AUC for every image
    - Per_BioRep: mean AUC per bio rep / time point / concentration (long format)
    - Per_Image_BioRep_Avg: one row per time point / concentration, one column per bio rep
    - TP_<time point>: all images for that time point, formatted for the Mixed Models converter
      (Column A = Filename, Column B = Bio Rep, Column C = AUC)
    """

    import os
    import cv2
    import numpy as np
    import pandas as pd
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    trapezoid = getattr(np, "trapezoid", None) or np.trapz

    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    # File pairs
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(set(c0.keys()) & set(c3.keys()))
    if not keys:
        raise ValueError("No C0/C3 pairs found")

    treatment_order = ["Ctrl", "5ng", "100ng", "500ng", "5ug"]

    image_rows = []

    print(f"\nProcessing {len(keys)} image pairs...\n")

    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])

        stem = os.path.splitext(c0[key])[0]
        tokens = stem.split("_")
        if len(tokens) < 5:
            print(f"  ⊗ Skipping {stem} - invalid filename format")
            continue

        bio_rep = tokens[0]
        time_point = tokens[1]
        if tokens[2] == "Tri" and tokens[3] == "Ctrl":
            treatment = "Ctrl"
        else:
            treatment = tokens[2]

        soma_xy = calculate_center_of_mass(c0_path, c3_path, stem)
        if soma_xy is None:
            print(f"  ✗ No center of mass: {stem}")
            continue
        cx, cy = soma_xy

        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"  ✗ Failed to load: {stem}")
            continue

        skel = skeletonize(img > threshold_li(img))

        # Continuous sampling (1 pixel step)
        n_steps = int(np.floor((max_radius_microns - start_radius_microns) / scale_microns_per_pixel)) + 1
        radii_um = start_radius_microns + np.arange(n_steps) * scale_microns_per_pixel
        radii_um = radii_um[radii_um <= max_radius_microns]
        radii_px = np.unique(np.round(radii_um / scale_microns_per_pixel).astype(int))
        if len(radii_px) < 2:
            continue

        radii_um_actual = radii_px * scale_microns_per_pixel
        counts = np.array(_count_intersections(skel, cx, cy, radii_px), dtype=float)

        auc = float(trapezoid(counts, radii_um_actual))

        image_rows.append({
            "Image": stem,
            "Bio Rep": bio_rep,
            "Time Point": time_point,
            "Treatment": treatment,
            "AUC": auc,
        })
        print(f"  → {stem}: AUC = {auc:.2f}")

    if not image_rows:
        raise ValueError("No images were successfully processed")

    def _add_tp(df):
        df["_tp"] = df["Time Point"].str.extract(r"(\d+)")[0].astype(float)
        return df

    # Per image
    df_img = pd.DataFrame(image_rows)
    df_img["Treatment"] = pd.Categorical(df_img["Treatment"], categories=treatment_order, ordered=True)
    df_img = _add_tp(df_img)
    df_img = df_img.sort_values(["_tp", "Treatment", "Bio Rep", "Image"]).drop(columns="_tp")

    # Per bio rep (long format)
    df_rep = (
        df_img.groupby(["Time Point", "Treatment", "Bio Rep"], observed=True)["AUC"]
        .agg(Mean_AUC="mean", N_Images="count")
        .reset_index()
    )
    df_rep = _add_tp(df_rep)
    df_rep = df_rep.sort_values(["_tp", "Treatment", "Bio Rep"]).drop(columns="_tp")

    # Bio rep averages in wide format (one row per time point / treatment)
    df_rep_wide = (
        df_rep.pivot_table(
            index=["Time Point", "Treatment"],
            columns="Bio Rep",
            values="Mean_AUC",
            observed=True,
        )
        .add_prefix("BioRep_")
        .reset_index()
    )
    biorep_cols = [c for c in df_rep_wide.columns if str(c).startswith("BioRep_")]
    df_rep_wide["Mean_Across_BioReps"] = df_rep_wide[biorep_cols].mean(axis=1)
    df_rep_wide = _add_tp(df_rep_wide)
    df_rep_wide = df_rep_wide.sort_values(["_tp", "Treatment"]).drop(columns="_tp")

    # Export
    excel_path = os.path.join(output_folder, excel_name)
    with pd.ExcelWriter(excel_path) as writer:
        df_img.to_excel(writer, sheet_name="Per_Image", index=False)
        df_rep.to_excel(writer, sheet_name="Per_BioRep", index=False)
        df_rep_wide.to_excel(writer, sheet_name="Per_Image_BioRep_Avg", index=False)

        # One tab per time point, formatted for the Mixed Models converter:
        # Column A = Filename, Column B = Bio Rep, Column C = AUC
        time_points = sorted(
            df_img["Time Point"].unique(),
            key=lambda t: float("".join(ch for ch in t if ch.isdigit() or ch == ".") or 0),
        )
        for tp in time_points:
            df_tp = df_img[df_img["Time Point"] == tp].copy()
            df_tp["Bio Rep"] = pd.to_numeric(df_tp["Bio Rep"], errors="coerce")
            # Remove the leading bio rep token (e.g., "1_6hr_100ng_10_4_C_0" -> "6hr_100ng_10_4_C_0")
            df_tp["Image"] = df_tp["Image"].str.split("_", n=1).str[1]
            # Normalize "Tri_Ctrl" -> "Ctrl" so the converter parses the treatment correctly
            df_tp["Image"] = df_tp["Image"].str.replace("Tri_Ctrl_", "Ctrl_", regex=False)
            df_tp = (
                df_tp[["Image", "Bio Rep", "AUC"]]
                .rename(columns={"Image": "Filename"})
                .sort_values("Filename", kind="mergesort")
            )
            df_tp.to_excel(writer, sheet_name=f"TP_{tp}"[:31], index=False)
    print(f"\n✓ Saved AUC results: {excel_path}")
    return df_rep_wide

In [ ]:
auc_summary = Calculate_Sholl_AUC(
    input_folder="/Users/allisonpickle/Desktop/All/Isolated_Microglia_DAPI",
    start_radius_microns=0,
    max_radius_microns=50,
    scale_microns_per_pixel=0.335,
    output_folder_name="72hr_Rep2",
)
auc_summary


Processing 219 image pairs...

  → 100ng_LPS_01_1_C_0: AUC = 130.32
  → 100ng_LPS_01_2_C_0: AUC = 10.55
  → 100ng_LPS_01_3_C_0: AUC = 210.38
  → 100ng_LPS_01_5_C_0: AUC = 13.90
  → 100ng_LPS_02_1_C_0: AUC = 35.34
  → 100ng_LPS_02_2_C_0: AUC = 67.67
  → 100ng_LPS_02_3_C_0: AUC = 154.10
  → 100ng_LPS_02_4_C_0: AUC = 104.86
  → 100ng_LPS_02_5_C_0: AUC = 126.63
  → 100ng_LPS_03_4_C_0: AUC = 82.91
  → 100ng_LPS_03_5_C_0: AUC = 243.55
  → 100ng_LPS_04_1_C_0: AUC = 231.49
  → 100ng_LPS_04_2_C_0: AUC = 190.11
  → 100ng_LPS_04_4_C_0: AUC = 329.64
  → 100ng_LPS_05_1_C_0: AUC = 423.44
  → 100ng_LPS_05_2_C_0: AUC = 248.57
  → 100ng_LPS_05_3_C_0: AUC = 60.97
  → 100ng_LPS_05_4_C_0: AUC = 372.52
  → 100ng_LPS_06_1_C_0: AUC = 186.09
  → 100ng_LPS_06_2_C_0: AUC = 305.52
  → 100ng_LPS_06_3_C_0: AUC = 111.56
  → 100ng_LPS_06_4_C_0: AUC = 48.41
  → 100ng_LPS_06_5_C_0: AUC = 351.58
  → 100ng_LPS_07_2_C_0: AUC = 130.82
  → 100ng_LPS_07_4_C_0: AUC = 130.31
  → 100ng_LPS_07_5_C_0: AUC = 179.39
  → 100ng_LPS

Bio Rep,Time Point,Treatment,Mean_Across_BioReps


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.filters import threshold_li
from skimage.morphology import skeletonize

def Calculate_Sholl_AUC_ByConcentration(
    input_folder,
    start_radius_microns=0,
    max_radius_microns=100,
    scale_microns_per_pixel=0.335,
    output_folder_name="Sholl Analysis - AUC by Concentration",
    excel_name="Sholl_AUC_Results_ByConcentration.xlsx",
):
    """
    Calculates Sholl AUC (continuous sampling) for each time point and concentration.
    Excel sheets:
    - Per_Image: AUC for every image
    - Per_BioRep: mean AUC per bio rep / time point / concentration (long format)
    - Per_Image_BioRep_Avg: one row per time point / concentration, one column per bio rep
    - Conc_<concentration>: all images (all time points) for that concentration,
      formatted for the Mixed Models converter
      (Column A = Filename, Column B = Bio Rep, Column C = AUC)
    """

    import os
    import cv2
    import numpy as np
    import pandas as pd
    from skimage.filters import threshold_li
    from skimage.morphology import skeletonize

    trapezoid = getattr(np, "trapezoid", None) or np.trapz

    output_folder = os.path.join(input_folder, output_folder_name)
    os.makedirs(output_folder, exist_ok=True)

    # File pairs
    c0 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_0.tiff")}
    c3 = {fn.split("_C_", 1)[0]: fn for fn in os.listdir(input_folder) if fn.endswith("_C_3.tiff")}
    keys = sorted(set(c0.keys()) & set(c3.keys()))
    if not keys:
        raise ValueError("No C0/C3 pairs found")

    treatment_order = ["Ctrl", "5ng", "100ng", "500ng", "5ug"]

    image_rows = []

    print(f"\nProcessing {len(keys)} image pairs...\n")

    for key in keys:
        c0_path = os.path.join(input_folder, c0[key])
        c3_path = os.path.join(input_folder, c3[key])

        stem = os.path.splitext(c0[key])[0]
        tokens = stem.split("_")
        if len(tokens) < 5:
            print(f"  ⊗ Skipping {stem} - invalid filename format")
            continue

        bio_rep = tokens[0]
        time_point = tokens[1]
        if tokens[2] == "Tri" and tokens[3] == "Ctrl":
            treatment = "Ctrl"
        else:
            treatment = tokens[2]

        soma_xy = calculate_center_of_mass(c0_path, c3_path, stem)
        if soma_xy is None:
            print(f"  ✗ No center of mass: {stem}")
            continue
        cx, cy = soma_xy

        img = cv2.imread(c0_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            print(f"  ✗ Failed to load: {stem}")
            continue

        skel = skeletonize(img > threshold_li(img))

        # Continuous sampling (1 pixel step)
        n_steps = int(np.floor((max_radius_microns - start_radius_microns) / scale_microns_per_pixel)) + 1
        radii_um = start_radius_microns + np.arange(n_steps) * scale_microns_per_pixel
        radii_um = radii_um[radii_um <= max_radius_microns]
        radii_px = np.unique(np.round(radii_um / scale_microns_per_pixel).astype(int))
        if len(radii_px) < 2:
            continue

        radii_um_actual = radii_px * scale_microns_per_pixel
        counts = np.array(_count_intersections(skel, cx, cy, radii_px), dtype=float)

        auc = float(trapezoid(counts, radii_um_actual))

        image_rows.append({
            "Image": stem,
            "Bio Rep": bio_rep,
            "Time Point": time_point,
            "Treatment": treatment,
            "AUC": auc,
        })
        print(f"  → {stem}: AUC = {auc:.2f}")

    if not image_rows:
        raise ValueError("No images were successfully processed")

    def _add_tp(df):
        df["_tp"] = df["Time Point"].str.extract(r"(\d+)")[0].astype(float)
        return df

    # Per image
    df_img = pd.DataFrame(image_rows)
    df_img["Treatment"] = pd.Categorical(df_img["Treatment"], categories=treatment_order, ordered=True)
    df_img = _add_tp(df_img)
    df_img = df_img.sort_values(["_tp", "Treatment", "Bio Rep", "Image"]).drop(columns="_tp")

    # Per bio rep (long format)
    df_rep = (
        df_img.groupby(["Time Point", "Treatment", "Bio Rep"], observed=True)["AUC"]
        .agg(Mean_AUC="mean", N_Images="count")
        .reset_index()
    )
    df_rep = _add_tp(df_rep)
    df_rep = df_rep.sort_values(["_tp", "Treatment", "Bio Rep"]).drop(columns="_tp")

    # Bio rep averages in wide format (one row per time point / treatment)
    df_rep_wide = (
        df_rep.pivot_table(
            index=["Time Point", "Treatment"],
            columns="Bio Rep",
            values="Mean_AUC",
            observed=True,
        )
        .add_prefix("BioRep_")
        .reset_index()
    )
    biorep_cols = [c for c in df_rep_wide.columns if str(c).startswith("BioRep_")]
    df_rep_wide["Mean_Across_BioReps"] = df_rep_wide[biorep_cols].mean(axis=1)
    df_rep_wide = _add_tp(df_rep_wide)
    df_rep_wide = df_rep_wide.sort_values(["_tp", "Treatment"]).drop(columns="_tp")

    # Export
    excel_path = os.path.join(output_folder, excel_name)
    with pd.ExcelWriter(excel_path) as writer:
        df_img.to_excel(writer, sheet_name="Per_Image", index=False)
        df_rep.to_excel(writer, sheet_name="Per_BioRep", index=False)
        df_rep_wide.to_excel(writer, sheet_name="Per_Image_BioRep_Avg", index=False)

        # One tab per concentration, containing all time points.
        # Column A = Filename (bio rep token removed), Column B = Bio Rep, Column C = AUC
        concentrations = [t for t in treatment_order if t in set(df_img["Treatment"].astype(str))]
        concentrations += sorted(set(df_img["Treatment"].astype(str)) - set(concentrations))

        for conc in concentrations:
            df_c = df_img[df_img["Treatment"].astype(str) == conc].copy()
            if df_c.empty:
                continue

            df_c["Bio Rep"] = pd.to_numeric(df_c["Bio Rep"], errors="coerce")
            # Remove the leading bio rep token (e.g., "1_6hr_100ng_10_4_C_0" -> "6hr_100ng_10_4_C_0")
            df_c["Image"] = df_c["Image"].str.split("_", n=1).str[1]
            # Normalize "Tri_Ctrl" -> "Ctrl"
            df_c["Image"] = df_c["Image"].str.replace("Tri_Ctrl_", "Ctrl_", regex=False)
            df_c = _add_tp(df_c)
            df_c = (
                df_c.sort_values(["_tp", "Bio Rep", "Image"], kind="mergesort")
                [["Image", "Bio Rep", "AUC"]]
                .rename(columns={"Image": "Filename"})
            )
            df_c.to_excel(writer, sheet_name=f"Conc_{conc}"[:31], index=False)

    print(f"\n✓ Saved AUC results: {excel_path}")
    return df_rep_wide

In [ ]:
auc_summary = Calculate_Sholl_AUC_ByConcentration(
    input_folder='/Users/allisonpickle/Desktop/Super LPS/3 BioReps Analysis/0_All_Bio_Reps/Sholl Analysis/Microglia_DAPI_Images',
    start_radius_microns=0,
    max_radius_microns=50,
    scale_microns_per_pixel=0.335,
)


Processing 3099 image pairs...

  → 1_24hr_100ng_01_1_C_0: AUC = 62.31
  → 1_24hr_100ng_01_2_C_0: AUC = 83.08
  → 1_24hr_100ng_01_3_C_0: AUC = 33.84
  → 1_24hr_100ng_01_4_C_0: AUC = 43.88
  → 1_24hr_100ng_01_5_C_0: AUC = 60.97
  → 1_24hr_100ng_02_1_C_0: AUC = 31.99
  → 1_24hr_100ng_02_2_C_0: AUC = 36.85
  → 1_24hr_100ng_02_3_C_0: AUC = 35.01
  → 1_24hr_100ng_03_1_C_0: AUC = 1.17
  → 1_24hr_100ng_03_2_C_0: AUC = 43.89
  → 1_24hr_100ng_03_3_C_0: AUC = 94.30
  → 1_24hr_100ng_03_4_C_0: AUC = 96.48
  → 1_24hr_100ng_03_5_C_0: AUC = 28.48
  → 1_24hr_100ng_04_1_C_0: AUC = 47.91
  → 1_24hr_100ng_04_2_C_0: AUC = 36.52
  → 1_24hr_100ng_04_3_C_0: AUC = 48.07
  → 1_24hr_100ng_04_4_C_0: AUC = 118.09
  → 1_24hr_100ng_04_5_C_0: AUC = 5.03
  → 1_24hr_100ng_05_2_C_0: AUC = 112.56
  → 1_24hr_100ng_05_3_C_0: AUC = 13.74
  → 1_24hr_100ng_05_4_C_0: AUC = 59.63
  → 1_24hr_100ng_05_5_C_0: AUC = 8.54
  → 1_24hr_100ng_06_1_C_0: AUC = 137.52
  → 1_24hr_100ng_06_2_C_0: AUC = 10.22
  → 1_24hr_100ng_06_3_C_0: AUC 